# Gaze Driven Scene Understanding

## Setup

In [ ]:
USE_SAM = True

import cv2
import torch
import base64

import numpy as np

import pandas as pd
pd.set_option('display.max_columns', None)

import pickle
# Display the structure of the mask data
from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt


if USE_SAM:
    import supervision as sv
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator


import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image


# palettes: deep, muted, pastel, bright, colorblind
sns.set_theme(style="white", context="paper", palette="bright")

from io import BytesIO


import pyimgur

import warnings

from openai import OpenAI

from collections import Counter
import ast

import pickle
import time

import os

import scipy.ndimage as ndimage

color_palette = [
    "#85c1e9",  # Sky Blue
    "#82e0aa",  # Soft Green
    "#f1948a",  # Coral
    "#c39bd3",  # Lavender
    "#f7dc6f",  # Sunshine Yellow
    "#f0b27a",  # Light Orange
    "#a2d9ce",  # Mint Green
    "#bfc9ca",  # Steel Gray
    "#fad7a0",  # Peach
    "#76d7c4"   # Teal
]

In [ ]:
# read openai api key
with open('API_Keys/openai.txt', 'r') as file:
    OPENAI_API_KEY = file.read().strip()

# read imgur api key
with open('API_Keys/imgur.txt', 'r') as file:
    CLIENT_ID = file.read().strip()


### Auxiliary Functions

In [ ]:


from sys import breakpointhook
os.chdir(HOME)

def load_image( image_path, img_type = "rgb", show_image = True, fig_size = (8,6) ):
    """
    Loads an image from the given path, optionally displays it.

    Args:
        image_path (str): The path to the image file.
        show_image (bool, optional): Whether to display the image using matplotlib. Defaults to True.

    Returns:
        numpy.ndarray: The loaded image as a NumPy array in RGB format.
    """

    image = cv2.imread(image_path)

    # Convert image from BGR (OpenCV default) to RGB
    if img_type == "rgb":
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    if show_image:
        plt.grid(False)
        plt.axis('off')
        plt.gcf().set_size_inches(fig_size[0],fig_size[1])
        plt.imshow(image)

    return image

def plot_fixations(image_path, gaze_data, figsize=(10, 6), title="Gaze Plot",
                         downsample_factor=10, alpha=0.6, linewidth=0.5, show_gaze=True,
                         edgecolors="w", color="red", plotDuration=True, durationSize=10, clusters=False):
    """
    Plots gaze fixations on an image.

    Args:
        image_path (str): Path to the image file.
        gaze_data (pd.DataFrame): A DataFrame containing gaze coordinates (X, Y) and optional fixation duration.
        figsize (tuple, optional): Figure size (width, height) in inches. Defaults to (10, 6).
        title (str, optional): Title of the plot. Defaults to "Gaze Plot".
        downsample_factor (int, optional): Factor to downsample gaze data for plotting. Defaults to 10.
        alpha (float, optional): Transparency of the fixation markers. Defaults to 0.6.
        linewidth (float, optional): Width of the fixation marker outlines. Defaults to 0.5.
        edgecolors (str, optional): Color of the fixation marker outlines. Defaults to "w" (white).
        color (str, optional): Color of the fixation markers. Defaults to "red".
    """
    # Load the image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img_height, img_width = image.shape[:2]

    if plotDuration:
        duration = gaze_data['FixationDuration'] / downsample_factor
    else:
        duration = durationSize

    # Filter gaze data to keep points within the image bounds
    gaze_data = gaze_data[gaze_data['X'] <= img_width]
    gaze_data = gaze_data[gaze_data['Y'] <= img_height]

    if clusters and 'Cluster' in gaze_data.columns:
        # If clusters are provided, create a color map for them
        gaze_data['Cluster'] = gaze_data['Cluster'].astype(int)
        palette = sns.color_palette("deep", len(gaze_data['Cluster'].unique()))
        color_map = {cluster: palette[cluster] for cluster in gaze_data['Cluster'].unique()}
        colors = gaze_data['Cluster'].map(color_map)
    else:
        colors = color

    if show_gaze:
        plt.figure(figsize=figsize)
        plt.imshow(image)
        
        # Plot gaze points with or without clusters
        plt.scatter(gaze_data['X'], gaze_data['Y'], c=colors, s=duration,
                    alpha=alpha, edgecolors=edgecolors, linewidth=linewidth)

        plt.title(title)
        plt.grid(False)
        plt.axis('off')
        plt.show()
    
    return image, gaze_data

def remove_initial_data(df, seconds=2, rows_per_second=3):
    """
    Removes the first few seconds of data for each participant in the DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame containing eye-tracking data.
        seconds (int): Number of seconds to remove from the start for each participant.
        rows_per_second (int): Number of rows that approximate one second of data.

    Returns:
        pd.DataFrame: A DataFrame with the initial seconds of data removed for each participant.
    """
    # Calculate total rows to remove for each participant based on seconds and rows_per_second
    rows_to_remove = round(seconds * rows_per_second)

    # Group the DataFrame by 'ParticipantID' and apply a lambda function to drop the first few rows
    return df.groupby('ParticipantID').apply(lambda x: x.iloc[rows_to_remove:]).reset_index(drop=True)


def extract_masks(predictor, gaze_participant, gaze_index, gaze_type = "single_point"):
    """
    Extracts prediction masks, scores, and logits for a given gaze point from a participant's gaze data.

    This function retrieves a specific gaze point by its index from a DataFrame containing gaze data,
    then uses a prediction model to generate corresponding masks, scores, and logits based on the gaze coordinates.

    Args:
        predictor: An object with a 'predict' method that accepts point coordinates and labels, and outputs masks, scores, and logits.
        gaze_participant (pd.DataFrame): DataFrame containing gaze data for a participant. Must include columns 'X' and 'Y'.
        gaze_index (int): Index of the gaze point in the DataFrame to use for prediction.

    Returns:
        tuple: A tuple containing three elements:
            - masks: The predicted masks for the gaze point.
            - scores: The scores associated with the masks.
            - logits: The logits from the prediction.

    Raises:
        IndexError: If the gaze_index is out of range for the DataFrame.
        KeyError: If the expected 'X' or 'Y' columns are missing from the DataFrame.
    """

    if gaze_type == "single_point":
        try:
            # Extract X and Y coordinates
            X = gaze_participant.loc[gaze_index, "X"]
            Y = gaze_participant.loc[gaze_index, "Y"]

            # Prepare input data for the model
            input_point = np.array([[X, Y]], dtype=np.float32)
            input_label = np.array([1], dtype=np.int32)

        except KeyError as e:
            raise KeyError("DataFrame must contain 'X' and 'Y' columns.") from e
        except IndexError as e:
            raise IndexError("Provided gaze_index is out of range for the DataFrame.") from e
    else:
        sample = gaze_participant[ gaze_participant['Indx'] == gaze_index ][['X', 'Y']]
        input_point = np.array(sample.values, dtype=np.float32)
        input_label = np.array([1]*len(input_point), dtype=np.int32)

        # print("Creating a mask with ", len(input_point), " data points")

    # Predict masks, scores, and logits using the predictor
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True,
    )

    return masks, scores, logits



def crop_to_mask(image, mask, threshold =0, show_image=False, output_image_path="cropped_image_with_alpha", title="Cropped Image with Mask", figsize=(10, 5)):

    if image.shape[:2] != mask.shape:
        raise ValueError("The mask and image must have the same dimensions.")

    y_dim, x_dim, c_dim = image.shape

    mask_non_zero = mask > 0
    mask_non_zero = preprocess_mask( mask_non_zero )
    coords = np.argwhere(mask_non_zero)

    if len(coords) == 0:
        mask_non_zero = mask > 0
        coords = np.argwhere(mask_non_zero)

    y_min, x_min = np.min(coords, axis=0)
    y_max, x_max = np.max(coords, axis=0)

    # Crop the image to the bounding box
    
    scaled_ymin = int(y_min - threshold if y_min - threshold > 0 else y_min)
    scaled_xmin = int(x_min - threshold if x_min - threshold > 0 else x_min)
    scaled_ymax = int(y_max + threshold if y_max + threshold + 1 < y_dim else y_max + 1)
    scaled_xmax = int(x_max + threshold if x_max + threshold + 1 < x_dim else x_max + 1)
    
    cropped_image = image[scaled_ymin:scaled_ymax, scaled_xmin:scaled_xmax]
    cropped_mask = mask_non_zero[scaled_ymin:scaled_ymax, scaled_xmin:scaled_xmax]

    # save the RGBA image
    Image.fromarray(cropped_image).save(output_image_path.replace("image_with_alpha", "mask") + "_thre_" + str(threshold) + ".png")
    
    # Create a new RGBA image from the cropped image
    cropped_image_with_alpha = np.zeros((cropped_image.shape[0], cropped_image.shape[1], 4), dtype=np.uint8)
    cropped_image_with_alpha[..., :3] = cropped_image
    cropped_image_with_alpha[..., 3] = cropped_mask * 255  # Mask to alpha channel conversion

    # Optionally save the RGBA image
    Image.fromarray(cropped_image_with_alpha).save(output_image_path + ".png")

    # save mask to pickle
    with open(output_image_path + ".pkl", "wb") as f:
        pickle.dump(cropped_mask, f)

    if show_image:
        plt.figure(figsize=figsize)
        plt.imshow(cropped_image_with_alpha)
        plt.title(title)
        plt.axis('off')
        plt.show()

    return cropped_image_with_alpha, x_min, x_max, y_min, y_max


def delete_files(folder_path):
    """
    Deletes all PNG files from the specified folder.

    Args:
        folder_path (str): Path to the folder from which PNG files will be deleted.

    Returns:
        int: The number of files deleted.
    """
    # Check if the folder path exists
    if not os.path.exists(folder_path):
        print(f"The specified folder does not exist: {folder_path}")
        return 0

    # Initialize counter for the number of files deleted
    files_deleted = 0

    # Iterate over each file in the directory
    for filename in os.listdir(folder_path):
        # Construct full file path
        file_path = os.path.join(folder_path, filename)
        # Check if the file is a PNG file
        if filename.lower().endswith('.png'):
            # Delete the file
            os.remove(file_path)
            files_deleted += 1
        if filename.lower().endswith('.pkl'):
            # Delete the file
            os.remove(file_path)
            files_deleted += 1

    print(f"Total files deleted: {files_deleted}")
    return files_deleted


def process_gaze(gaze_participant, gaze_type = "single_point", circ_num_points = 5,
                    circ_diameter = 44, arm_length_horizontal = 22, arm_length_vertical = 22, horizontal = 11, vertical = 11):

    if gaze_type == "single_point":
        gaze_participant["Indx"] = gaze_participant.index

    if gaze_type == "circle":
        gaze_participant = sample_points_circle(gaze_participant, circ_num_points, diameter=circ_diameter)

    if gaze_type == "cross_full":
        gaze_participant = sample_points_cross(gaze_participant, arm_length_horizontal=arm_length_horizontal,
                                            arm_length_vertical = arm_length_vertical)

    if gaze_type == "cross_upper":
        gaze_participant = sample_points_cross(gaze_participant, arm_length_horizontal=arm_length_horizontal,
                                            arm_length_vertical = arm_length_vertical,  remove_bottom = True)

    if gaze_type == "box":
        gaze_participant = sample_points_box(gaze_participant, horizontal = horizontal, vertical = vertical)

    return gaze_participant


def apply_SAM_single_point(HOME, data_file, num_imgs, condition_id, center_bias_sec, gaze_type = "single_point", start_indx = 1,
                            exp_type="expected", delete_existing_files=False, show_gaze =False, show_output = True, show_masks = True,
                            circ_num_points = 5, circ_diameter = 44, arm_length_horizontal = 22, arm_length_vertical = 22, horizontal = 11, vertical = 11, threshold = 0 ):
    """
    Processes gaze data for a set number of images, applies gaze-driven mask prediction, and optionally displays or saves the output.

    Args:
        HOME (str): Base directory for input/output data.
        data_file (str): Path to the CSV file containing gaze data, relative to HOME.
        num_imgs (int): Number of images to process.
        condition_id (int): ID of the condition to filter the data.
        center_bias_sec (int): Number of seconds to remove from the start of each participant's data to reduce center bias.
        exp_type (str): Type of experiment ('expected' or 'unexpected'), which affects input/output paths. Defaults to 'expected'.
        delete_existing_files (bool): If True, deletes existing files in the output directory before processing. Defaults to False.
        show_output (bool): If True, displays the output images. Defaults to False.

    Effects:
        Processes each image according to the specified conditions, predicts masks based on gaze data, and manages output files based on user options.
    """
    # Configuration and file cleanup
    if delete_existing_files:
        delete_files(f"{HOME}/gaze_driven_masks/{gaze_type}/{exp_type}/")

    data = pd.read_csv(f"{HOME}/{data_file}")

    # Processing each image
    for IMG_ID in range(start_indx, num_imgs + 1):
        print(f"Processing Image {IMG_ID} of {num_imgs}")

        # Filtering and adjusting data for center bias
        filtered_data = data[(data['ItemNum'] == IMG_ID) & (data['Condition'] == condition_id)]
        gaze_participant = filtered_data[['ParticipantID', 'X', 'Y', 'FixationDuration']].copy()
        gaze_participant.reset_index(drop=True, inplace=True)
        gaze_participant = remove_initial_data(gaze_participant, seconds=center_bias_sec)
        print("Data size after removing bias:", gaze_participant.shape)

        gaze_participant = process_gaze(gaze_participant, gaze_type = gaze_type, circ_num_points = circ_num_points,
                    circ_diameter = circ_diameter, arm_length_horizontal = arm_length_horizontal, arm_length_vertical = arm_length_vertical, horizontal = horizontal, vertical = vertical)

        print("Data size after sampling:", gaze_participant.shape)
        print("Unique gaze points:", len(gaze_participant["Indx"].unique()))

        # Image path and loading
        image_type_suffix = 'exp' if exp_type == "expected" else 'unexp'
        main_scene_path = f"{HOME}/exp_images/{IMG_ID}{image_type_suffix}.jpg"
        results_dir = f"{HOME}/gaze_driven_masks/{gaze_type}/{exp_type}/"

        image, gaze_participant = plot_fixations(main_scene_path, gaze_participant,
                                                show_gaze = show_gaze, plotDuration = False,
                                                durationSize = 10, figsize=(10, 6),
                                                title=f"Fixation Plot for Condition {condition_id} | IMG_ID {IMG_ID}", alpha=0.8)

        # Mask extraction and processing
        predictor = SAM2ImagePredictor(sam2_model)
        predictor.set_image(image)

        results_df = initialize_results_df(IMG_ID, main_scene_path, HOME, gaze_type)

        row_indx = 0
        for gz_indx in gaze_participant['Indx'].unique():

            masks, scores, logits = extract_masks(predictor, gaze_participant, gz_indx, gaze_type)

            results_df.loc[row_indx, "IMG_ID"] = IMG_ID
            results_df.loc[row_indx, "IMG_PATH"] = main_scene_path
            results_df.loc[row_indx, "GAZE_INDX"] = gz_indx
            results_df.loc[row_indx, "GAZE_X"] = gaze_participant.iloc[gz_indx]['X']
            results_df.loc[row_indx, "GAZE_Y"] = gaze_participant.iloc[gz_indx]['Y']

            results_df = generate_metadata(image, main_scene_path, results_df, masks, scores, row_indx, gaze_participant, gaze_type, gz_indx,  IMG_ID, results_dir, show_output = show_output, show_masks = show_masks, threshold = threshold)
            row_indx = row_indx + 1

        results_df.to_csv(f"{HOME}/gaze_driven_masks/{gaze_type}/metadata/IMG_{IMG_ID}.csv", index=False)

        # wait 5 seconds
        time.sleep(5)

    return results_df


def sample_points_circle(df, N, diameter = 10):
    sampled_points = []

    for index, row in df.iterrows():
        x_center = row['X']
        y_center = row['Y']
        radius = diameter / 2 #(row['FixationDuration'] / downsample_factor)
        participantID = row['ParticipantID']
        fixationDur = row['FixationDuration']

        sampled_points.append((participantID, x_center, y_center, fixationDur, index))

        # Generate N points at regular intervals around the circle
        for i in range(N):
            angle = 2 * np.pi * i / N  # Regular interval
            x = x_center + radius * np.cos(angle)
            y = y_center + radius * np.sin(angle)
            sampled_points.append((participantID, x, y, fixationDur, index))


    return pd.DataFrame(sampled_points, columns=['ParticipantID', 'X', 'Y', 'FixationDuration', 'Indx'])


def sample_points_cross(df, arm_length_horizontal=11, arm_length_vertical = 11,  remove_bottom = False, remove_top = False, remove_left = False, remove_right = False):
    sampled_points = []

    for index, row in df.iterrows():
        x_center = row['X']
        y_center = row['Y']

        participantID = row['ParticipantID']
        fixationDur = row['FixationDuration']

        # Points at the tips of the cross
        # Horizontal arm tips
        x_right = x_center + arm_length_horizontal
        x_left = x_center - arm_length_horizontal

        # Vertical arm tips
        y_bottom= y_center + arm_length_vertical
        y_top = y_center - arm_length_vertical

        sampled_points.append((participantID, x_center, y_center, fixationDur, index))


        # Append points to the list
        if not remove_right:
            sampled_points.append((participantID, x_right, y_center, fixationDur, index))

        if not remove_left:
            sampled_points.append((participantID, x_left, y_center, fixationDur, index))

        if not remove_top:
            sampled_points.append((participantID, x_center, y_top, fixationDur, index))

        if not remove_bottom:
            sampled_points.append((participantID, x_center, y_bottom, fixationDur, index))


    return pd.DataFrame(sampled_points, columns=['ParticipantID', 'X', 'Y', 'FixationDuration', 'Indx'])


def sample_points_box(df, horizontal = 11, vertical = 11, center = True):
    sampled_points = []

    for index, row in df.iterrows():
        x_center = row['X']
        y_center = row['Y']

        participantID = row['ParticipantID']
        fixationDur = row['FixationDuration']

        # center point
        if center:
            sampled_points.append((participantID, x_center, y_center, fixationDur, index))
        
        # make a box
        x_min = x_center - horizontal
        x_max = x_center + horizontal
        y_min = y_center - vertical
        y_max = y_center + vertical
        
        sampled_points.append((participantID, x_min, y_min, fixationDur, index))
        sampled_points.append((participantID, x_max, y_min, fixationDur, index))
        sampled_points.append((participantID, x_min, y_max, fixationDur, index))
        sampled_points.append((participantID, x_max, y_max, fixationDur, index))
    
    return pd.DataFrame(sampled_points, columns=['ParticipantID', 'X', 'Y', 'FixationDuration', 'Indx'])



def upload_img_to_server(image_path, client_id):
    """
    Uploads an image to Imgur after verifying that the image file exists.

    Parameters:
    - client_id (str): The Client ID for the Imgur API.
    - image_path (str): The file path of the image to be uploaded.

    Returns:
    - str: URL of the uploaded image if successful, or an error message.
    """
    #print("Client ID", client_id)
    # Check if the image file exists
    if not os.path.isfile(image_path):
        return "Error: The file does not exist at the specified path."

    # Initialize the Imgur client with the provided client ID
    imgur = pyimgur.Imgur(client_id)

    try:
        # Attempt to upload the image to Imgur
        uploaded_image = imgur.upload_image(image_path, title="Uploaded via PyImgur")
        return uploaded_image.link  # Return the URL of the uploaded image
    except Exception as e:
        # Return the error message if upload fails
        return f"An error occurred during upload: {e}"

def classify_obj( obj_url, system_prompt, obj_prompt ):

    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
        "role": "system",
        "content": [
            {
            "type": "text",
            "text": system_prompt
            }
        ]
        },
        {
        "role": "user",
        "content": [
            {"type": "text", "text": obj_prompt},
            {
            "type": "image_url",
            "image_url": {
                "url": obj_url,
            },
            },
        ],
        }
    ],
    max_tokens=300,
    )

    return response.choices[0].message.content

def classify_obj_with_context( scene_img_url, mask_obj_url, system_prompt, scene_prompt, mask_prompt   ):

    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
        "role": "system",
        "content": [
            {
            "type": "text",
            "text": system_prompt
            }
        ]
        },
        {
        "role": "user",
        "content": [
            {
            "type": "text",
            "text": "This is the link to the main scene"
            },
            {
            "type": "image_url",
            "image_url": {
                "url": scene_img_url
            }
            },
            {
            "type": "text",
            "text": scene_prompt
            }
        ]
        },
        {
        "role": "user",
        "content": [
            {
            "type": "text",
            "text": mask_prompt
            },
            {
            "type": "image_url",
            "image_url": {
                "url": mask_obj_url
            }
            }
        ]
        },
    ],
    temperature=1,
    max_tokens=256,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0
    )

    return response.choices[0].message.content

def plot_processed_img(main_scene_path, gaze_participant, gaze_index, results_df, masks, gaze_type, row_indx, figsize = (15, 5)):

    num_masks = len(masks)
    prefix = get_prefix(gaze_type)

    fig, axs = plt.subplots(1, num_masks, figsize=figsize)  # Adjust figsize as needed

    for mask_indx in range(num_masks):
        columns = [f'{prefix}_MASK_{mask_indx}_XMIN', f'{prefix}_MASK_{mask_indx}_XMAX',
                f'{prefix}_MASK_{mask_indx}_YMIN', f'{prefix}_MASK_{mask_indx}_YMAX', f'{prefix}_MASK_{mask_indx}_AREA']

        xmin = results_df.loc[row_indx, f'{prefix}_MASK_{mask_indx}_XMIN']
        xmax = results_df.loc[row_indx, f'{prefix}_MASK_{mask_indx}_XMAX']
        ymin = results_df.loc[row_indx, f'{prefix}_MASK_{mask_indx}_YMIN']
        ymax = results_df.loc[row_indx, f'{prefix}_MASK_{mask_indx}_YMAX']
        area = results_df.loc[row_indx, f'{prefix}_MASK_{mask_indx}_AREA']

        # Select the current subplot
        ax = axs[mask_indx]
        plt.sca(ax)  # Set the current Axes instance to ax

        # Plot the point and mask on the image
        img, title = plot_point_on_image(main_scene_path, gaze_participant, gaze_index,
                            mask=masks[mask_indx],
                            bbox=[xmin, xmax, ymin, ymax],
                            mask_color=[255, 0, 0])
        
        # get x, y points
        
        if gaze_type == "single_point":
            x = gaze_participant[gaze_index, 'X']
            y = gaze_participant.loc[gaze_index, 'Y']
        else:
            x = gaze_participant.loc[:, 'X']
            y = gaze_participant.loc[:, 'Y']
            ax.scatter(x, y, color='green', marker='o', s=10)
        
        ax.imshow(img)

        # Optionally set titles or other aesthetics
        ax.set_title(title + f' AREA = {area}')
        ax.axis('off')  # Turn off axis if not needed

    plt.tight_layout()  # Adjust subplots to fit into figure area.
    plt.show()


def preprocess_mask(mask, size_threshold=100):

    # Label connected components
    labeled_mask, num_features = ndimage.label(mask)

    # Create a new mask where small components are removed
    component_sizes = np.bincount(labeled_mask.ravel())
    too_small = component_sizes < size_threshold
    too_small_mask = too_small[labeled_mask]

    # Zero out small components
    filtered_mask = mask.copy()
    filtered_mask[too_small_mask] = 0

    return filtered_mask

def plot_point_on_image(image_path, df, indx, mask=None, bbox=None, point_color=(0, 255, 0), mask_color=[0, 0, 255], bbox_color=(0, 0, 255)):
    """
    Load an image and plot a single point from DataFrame coordinates on it.
    Optionally overlay a mask and draw a bounding box.

    Args:
        image_path (str): Path to the input image.
        df (pandas.DataFrame): DataFrame with columns 'X' and 'Y'.
        indx (int): Index of the DataFrame for the point to plot.
        mask (numpy.ndarray, optional): Binary mask to overlay on the image.
        bbox (tuple, optional): Tuple of (xmin, xmax, ymin, ymax) for the bounding box.
        point_color (tuple): Color for the point.
        mask_color (list): Color for the mask overlay.
        bbox_color (tuple): Color for the bounding box.

    Returns:
        None: Displays the image with modifications.
    """
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError("The specified image path does not exist.")

    if indx >= len(df) or indx < 0:
        raise IndexError("The provided index is out of bounds.")

    # Ensure the DataFrame contains the necessary columns
    if 'X' not in df.columns or 'Y' not in df.columns:
        raise ValueError("DataFrame must contain 'X' and 'Y' columns.")

    # Extract point coordinates
    x, y = int(np.round(df['X'].iloc[indx])), int(np.round(df['Y'].iloc[indx]))
    print(x)
    print(y)

    # Draw the point
    cv2.circle(image, (x, y), radius=5, color=point_color, thickness=-1)

    # Overlay mask
    if mask is not None:
        if mask.shape != (image.shape[0], image.shape[1]):
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

        color_mask = np.zeros_like(image)
        color_mask[mask > 0.0001] = mask_color
        image = cv2.addWeighted(image, 0.8, color_mask, 0.2, 0)

    # Draw bounding box
    if bbox is not None:
        cv2.rectangle(image, (bbox[0], bbox[2]), (bbox[1], bbox[3]), color=bbox_color, thickness=2)

    if bbox is not None:
        title = f'Fixation = ({x}, {y}) | BBox = [{bbox[0]}, {bbox[2]}, {bbox[1]}, {bbox[3]}]'
    else:
        title = f'Fixation = ({x}, {y})'

    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), title

import pandas as pd

def get_prefix(gaze_type):

    prefix_dict = {
        "circle": "CRL",
        "box" : "BOX",
        "cross_full": "CRXFull",
        "cross_upper": "CRXUpper",
        "final" : "Final",
    }

    return prefix_dict.get(gaze_type, "GZP")

def initialize_results_df(img_id, main_scene_path, working_dir, gaze_type="single_point", num_masks=3, num_gpt_ans=10):

    prefix = get_prefix(gaze_type)

    # Initialize columns for DataFrame
    df_columns = ['IMG_ID', 'IMG_PATH', 'IMG_LINK', 'GAZE_INDX', 'GAZE_X', 'GAZE_Y']

    # Add mask and GPT answer columns
    for mask_id in range(num_masks):
        df_columns.extend([f'{prefix}_MASK_{mask_id}_LINK',
                            f'{prefix}_MASK_{mask_id}_ARRAY',
                            f'{prefix}_MASK_{mask_id}_IMG',
                            f'{prefix}_MASK_{mask_id}_SCORE',
                            f'{prefix}_MASK_{mask_id}_XMIN',
                            f'{prefix}_MASK_{mask_id}_XMAX',
                            f'{prefix}_MASK_{mask_id}_YMIN',
                            f'{prefix}_MASK_{mask_id}_YMAX',
                            f'{prefix}_MASK_{mask_id}_AREA'])
        for gpt_ans in range(num_gpt_ans):
            df_columns.append(f'{prefix}_MASK_{mask_id}_GPTANSWER_{gpt_ans}')

        if gaze_type == "final":
            break

    # Create DataFrame with specified columns
    results_df = pd.DataFrame(columns=df_columns)

    if gaze_type == "final":
        results_df['BEST_MASK_INDX'] = None
        results_df['BEST_MASK_TYPE'] = None

        results_df.rename(columns={'Final_MASK_0_LINK': 'BEST_MASK_LINK'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_ARRAY':'BEST_MASK_ARRAY'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_IMG':  'BEST_MASK_IMG'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_SCORE':'BEST_MASK_SCORE'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_XMIN': 'BEST_MASK_XMIN'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_XMAX': 'BEST_MASK_XMAX'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_YMIN': 'BEST_MASK_YMIN'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_YMAX': 'BEST_MASK_YMAX'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_AREA': 'BEST_MASK_AREA'}, inplace=True)

        results_df.rename(columns={'Final_MASK_0_GPTANSWER_0': 'BEST_MASK_GPTANSWER_0'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_1': 'BEST_MASK_GPTANSWER_1'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_2': 'BEST_MASK_GPTANSWER_2'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_3': 'BEST_MASK_GPTANSWER_3'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_4': 'BEST_MASK_GPTANSWER_4'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_5': 'BEST_MASK_GPTANSWER_5'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_6': 'BEST_MASK_GPTANSWER_6'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_7': 'BEST_MASK_GPTANSWER_7'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_8': 'BEST_MASK_GPTANSWER_8'}, inplace=True)
        results_df.rename(columns={'Final_MASK_0_GPTANSWER_9': 'BEST_MASK_GPTANSWER_9'}, inplace=True)

    return results_df

def encode_image(filepath, new_width=800, new_height=600):
    # Open the image and resize it
    with Image.open(filepath) as img:
        img = img.resize((new_width, new_height), Image.ANTIALIAS)  # Resize the image

        # Create a BytesIO object to save image to bytes
        img_byte_arr = BytesIO()
        img.save(img_byte_arr, format='JPEG')  # Save your image to byte array
        img_byte_arr = img_byte_arr.getvalue()

    # Encode bytes to Base64
    encoded = base64.b64encode(img_byte_arr).decode('utf-8')
    return "data:image/jpg;base64," + encoded


def generate_metadata(image, main_scene_path, results_df, masks, scores, row_indx, gaze_participant, gaze_type, gaze_index,  img_id, results_dir,threshold = 0, show_output = True, show_masks = True):

    figsize = (10, 5)
    masks_processed = []
    prefix = get_prefix(gaze_type)

    for mask_id in range(0, len(masks)):

        mask = masks[mask_id]
        score = scores[mask_id]

        output_mask_path = f"{results_dir}/IMG_{img_id}_MaskIndx_{gaze_index}_Output_{mask_id}_Score_{score:.4f}"
        mask_cropped, xmin, xmax, ymin, ymax = crop_to_mask(image, mask, threshold = threshold, show_image=False, output_image_path=output_mask_path, title=f"IMG_{img_id}_Mask_{mask_id}_Score_{score}", figsize=figsize)
        area = abs(xmax-xmin) * (ymax-ymin)

        # saved masks
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_ARRAY"] = output_mask_path + ".pkl"
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_IMG"] = output_mask_path + ".png"

        # BBox Info
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_SCORE"] = score
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_XMIN"] = xmin
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_XMAX"] = xmax
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_YMIN"] = ymin
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_YMAX"] = ymax
        results_df.loc[row_indx, f"{prefix}_MASK_{mask_id}_AREA"] = area

        masks_processed.append(mask_cropped)

    if show_output:

        plot_processed_img(main_scene_path, gaze_participant, gaze_index, results_df, masks, gaze_type, row_indx)

        if show_masks:
            sv.plot_images_grid(
                images=masks_processed,
                titles=[f"score: {score:.4f}" for score in scores],
                grid_size=(1, len(masks_processed)),
                size = figsize)

    return results_df


def find_best_mask( metadata, results_df, score_cols, prefix_to_gazetype,  indx, img_id, CLIENT_ID ):
    
    # find the mask with the highest score
    best_mask_score = -1
    best_mask_xmax = None
    best_mask_xmin = None
    best_mask_ymax = None
    best_mask_ymin = None
    best_mask_area = None

    for i in range(0, len(score_cols)):

        if metadata.loc[indx, score_cols[i]] > best_mask_score:
            best_mask_score = metadata.loc[indx, score_cols[i]]
            best_indx = score_cols[i].split("_")[2]
            prefix = score_cols[i].split("_")[0]
            results_df.loc[indx, "BEST_MASK_INDX"] = best_indx
            results_df.loc[indx, "BEST_MASK_TYPE"] = prefix_to_gazetype[ prefix ]
            best_mask_xmin = f"{prefix}_MASK_{best_indx}_XMIN"
            best_mask_xmax = f"{prefix}_MASK_{best_indx}_XMAX"
            best_mask_ymin = f"{prefix}_MASK_{best_indx}_YMIN"
            best_mask_ymax = f"{prefix}_MASK_{best_indx}_YMAX"
            best_mask_area = f"{prefix}_MASK_{best_indx}_AREA"
            best_mask_array = f"{prefix}_MASK_{best_indx}_ARRAY"
            best_mask_img = f"{prefix}_MASK_{best_indx}_IMG"

    results_df.loc[indx, "BEST_MASK_SCORE"] = best_mask_score
    results_df.loc[indx, "BEST_MASK_XMIN"] = metadata.loc[indx, best_mask_xmin]
    results_df.loc[indx, "BEST_MASK_XMAX"] = metadata.loc[indx, best_mask_xmax]
    results_df.loc[indx, "BEST_MASK_YMIN"] = metadata.loc[indx, best_mask_ymin]
    results_df.loc[indx, "BEST_MASK_YMAX"] = metadata.loc[indx, best_mask_ymax]
    results_df.loc[indx, "BEST_MASK_AREA"] = metadata.loc[indx, best_mask_area]

    results_df.loc[indx,'BEST_MASK_ARRAY'] = metadata.loc[indx, best_mask_array]
    results_df.loc[indx,'BEST_MASK_IMG'] = metadata.loc[indx, best_mask_img].replace("//", "/")

    link =  results_df.loc[indx,'BEST_MASK_IMG']

    best_mask_link = upload_img_to_server( link )
    print("Response", best_mask_link)
    
    if "An error occurred during upload" in best_mask_link:
        print("\tTrying with single point")
        link_temp = link.replace(results_df.loc[indx, "BEST_MASK_TYPE"],  "single_point")
        best_mask_link = upload_img_to_server( link_temp, CLIENT_ID )

    if "An error occurred during upload" in best_mask_link:
        print(f"\t[UPLOAD ERROR] The uploading process {link} resulted in error -1. Trying again with a different CLIENT_ID")
        # change CLIENT_ID
        if CLIENT_ID == "52aa0e29e52a4a1":
            CLIENT_ID = "003e7d0cabc152b"
        else: 
            CLIENT_ID = "52aa0e29e52a4a1"
        best_mask_link = upload_img_to_server( link )

    if "An error occurred during upload" in best_mask_link:
        print("\t[EXCEPTION] Waiting 1h")
        
        results_df.to_csv(f"{HOME}/results/results_{img_id}_{indx}.csv", index = False)
        time.sleep(3700)
        best_mask_link = upload_img_to_server( link )

    results_df.loc[indx,'BEST_MASK_LINK'] = best_mask_link
    print("\tUploaded Mask", best_mask_link)

    return results_df


def get_rankings_from_scene( scene_img_url, system_prompt, ranking_prompt   ):
    
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
        "role": "system",
        "content": [
            {
            "type": "text",
            "text": system_prompt
            }
        ]
        },
        {
        "role": "user",
        "content": [
            {
            "type": "text",
            "text": "This is the link to the main scene"
            },
            {
            "type": "image_url",
            "image_url": {"url": scene_img_url}
            },
            {
            "type": "text",
            "text": ranking_prompt
            },
        ]
        },
    ],
    temperature=0.1,
    max_tokens=3000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0
    )

    return response.choices[0].message.content


def show_masks(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True):
    for i, (mask, score) in enumerate(zip(masks, scores)):
        plt.figure(figsize=(10, 10))
        plt.imshow(image)
        show_mask(mask, plt.gca(), borders=borders)
        if point_coords is not None:
            assert input_labels is not None
            show_points(point_coords, input_labels, plt.gca())
        if box_coords is not None:
            # boxes
            show_box(box_coords, plt.gca())
        if len(scores) > 1:
            plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()
        
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        import cv2
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=150):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))    
    
    
def show_anns(anns, borders=True):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.5]])
        img[m] = color_mask 
        if borders:
            import cv2
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1) 

    ax.imshow(img)
    
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
import numpy as np

# Define a function to cluster the eye gaze data
def cluster_eye_gazes(df, eps=25, min_samples=3):
    # Extract X and Y coordinates for clustering
    eye_gaze_coordinates = df[['X', 'Y']].values
    
    # Perform DBSCAN clustering
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    df['Cluster'] = dbscan.fit_predict(eye_gaze_coordinates)
    
    return df, dbscan

import networkx as nx
import matplotlib.pyplot as plt


def generate_markov_chain( img_id ):
    
    # Load the background image (the room image)
    image_path = f"{HOME}/exp_images/{img_id}exp.jpg"
    img = load_image( image_path, show_image = False )

    results_df = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")


    # Initialize an empty dictionary to store graphs for each participant
    participant_graphs = {}


    bounding_box_columns = ['MOST_FREQ_MASK', 'BEST_MASK_XMIN', 'BEST_MASK_XMAX', 'BEST_MASK_YMIN', 'BEST_MASK_YMAX']

    # Group the data by participant
    grouped_data = results_df.groupby('PARTICIPANT_ID')

    # Initialize an empty directed graph for all participants
    combined_graph = nx.DiGraph()

    # Iterate through each participant's data and combine the transitions into one graph
    for participant_id, group in grouped_data:
        # Extract the most frequent mask labels (nodes) in the order they appear
        gaze_sequence = group['MOST_FREQ_MASK'].tolist()
        
        # Add edges between consecutive gaze points (transitions between masks)
        for i in range(len(gaze_sequence) - 1):
            current_mask = gaze_sequence[i]
            next_mask = gaze_sequence[i + 1]
            if combined_graph.has_edge(current_mask, next_mask):
                # If the edge exists, increment the weight (transition count)
                combined_graph[current_mask][next_mask]['weight'] += 1
            else:
                # Otherwise, create the edge with an initial weight of 1
                combined_graph.add_edge(current_mask, next_mask, weight=1)


    # Normalize the weights (transition counts) to probabilities
    for node in combined_graph.nodes():
        total_outgoing = sum(d['weight'] for u, v, d in combined_graph.out_edges(node, data=True))
        if total_outgoing > 0:
            for u, v, d in combined_graph.out_edges(node, data=True):
                combined_graph[u][v]['weight'] = d['weight'] / total_outgoing  # convert to probabilities

    # Get the updated edge labels (probabilities)
    combined_edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in combined_graph.edges(data=True)}

    # Calculate the center (X, Y) of each bounding box
    results_df['center_x'] = (results_df['BEST_MASK_XMIN'] + results_df['BEST_MASK_XMAX']) / 2
    results_df['center_y'] = (results_df['BEST_MASK_YMIN'] + results_df['BEST_MASK_YMAX']) / 2

    # Group by the most frequent mask and take the mean of the center coordinates (in case there are multiple)
    mask_centers = results_df.groupby('MOST_FREQ_MASK')[['center_x', 'center_y']].mean()

    # Normalize the positions to fit into a suitable plotting space (since the coordinates are image-based)
    mask_centers['center_x'] = ( (mask_centers['center_x'] - mask_centers['center_x'].min()) / (mask_centers['center_x'].max() - mask_centers['center_x'].min()) ) * 0.95
    mask_centers['center_y'] = ( (mask_centers['center_y'] - mask_centers['center_y'].min()) / (mask_centers['center_y'].max() - mask_centers['center_y'].min()) ) * (0.95)

    # Convert to a dictionary format to be used in the graph plotting
    pos = {mask: (mask_centers.loc[mask, 'center_x'], 1 - mask_centers.loc[mask, 'center_y']) for mask in mask_centers.index}


    target_colors = {}
    for mask, group in results_df.groupby('MOST_FREQ_MASK'):
        if group['IS_TARGET'].max() == 1:  # If it's a target in any row, set to different color
            target_colors[mask] = '#FF6F61'  # A reddish color for the target nodes
        else:
            target_colors[mask] = '#AED6F1'  # Default light blue for non-targets


    node_colors = [target_colors[mask] if mask in target_colors else '#AED6F1' for mask in combined_graph.nodes()]
    
    weights = [combined_graph[u][v]['weight'] * 7 for u, v in combined_graph.edges()]

    # Now we can plot the graph using the bounding box center positions
    # Load the background image again and set it as a light alpha backdrop
    plt.figure(figsize=(14, 10))

    # Display the image with reduced opacity (alpha)
    plt.imshow(img, extent=[0, 1, 0, 1], alpha=0.5)

    # Normalize positions to fit into the image coordinate system (assuming 0 to 1 range)
    pos_scaled = {key: (value[0], value[1]) for key, value in pos.items()}

    # Draw the graph with the scaled positions
    nx.draw(combined_graph, pos_scaled, with_labels=True, node_size=2000, node_color=node_colors, font_size=12, font_weight='bold', font_color='black')

    # Draw edges with appropriate thickness and light gray color
    nx.draw_networkx_edges(combined_graph, pos_scaled, width=weights, edge_color='gray', arrowsize=15)

    # Draw edge labels
    nx.draw_networkx_edge_labels(combined_graph, pos_scaled, edge_labels=combined_edge_labels, font_color='black', font_size=8)

    # Turn off axis for cleaner presentation
    plt.axis('off')

    plt.tight_layout()
    plt.title("Markov Chain Extracted from Participants Gaze)", fontsize=18, pad=20)
    
    plt.savefig(f"{HOME}/results/Markov_Chains/{img_id}_markov_chain.png")
    
    # save the graph
    with open(f"{HOME}/results/Markov_Chains/{img_id}_markov_chain.pkl", 'wb') as f:
        pickle.dump(combined_graph, f)
    
    plt.show()
    




import networkx as nx

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from collections import Counter

def generate_image_description(img_id, model="gpt-4o"):
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    # Read the results file
    results_df = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")
    
    # Extract the main scene image link (assuming IMG_LINK column exists)
    scene_img_url = results_df.loc[0, 'IMG_LINK']
    print(scene_img_url)

    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
        "role": "system",
        "content": [{"type": "text", "text": """You are an agent that analyses scenes. You receive a scene and it is your job to describe the scene with the highest detail possible. 
                    Describe the objects of the scene in terms of their properties (ex: shape, color, functionality, etc.).
                    Describe the spatial relationships between the objects in the scene.
                    Describe your understanding of the functionality of the objects in the scene."""}]},
        {"role": "user", "content": [{ "type": "text", "text": "This is the link to the main scene"}, 
                                    { "type": "image_url", "image_url": { "url": scene_img_url, "detail": "high"}}]},
    ],
    )
    
    scene_description = response.choices[0].message.content
    print(scene_description)
    
    for indx in results_df.index:
        results_df.loc[indx, 'SCENE_DESCRIPTION'] = scene_description
    
    results_df.to_csv(f"{HOME}/results/results_{img_id}_with_context.csv", index = False)
    
    
def get_scene_description( img_id ):
    
    file_path = f"{HOME}/results/results_{img_id}_with_context.csv"
    df = pd.read_csv(file_path)
    scene_description = df.loc[0, 'SCENE_DESCRIPTION']
    
    return scene_description


def create_mask_dict(img_id, debug = True):
    
    df = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")
    
    # Initialize the dictionary to store the result
    mask_dict = {}
    
    # Iterate through the dataframe rows
    for index, row in df.iterrows():
        most_freq_mask = row['MOST_FREQ_MASK']
        gaze_indx = row['GAZE_INDX']
        
        # Create the bounding box array (assuming these columns exist)
        bbox = np.array([row['BEST_MASK_XMIN'], row['BEST_MASK_XMAX'], row['BEST_MASK_YMIN'], row['BEST_MASK_YMAX']])
        
        # If the most frequent mask is not yet in the dictionary, add it
        if most_freq_mask not in mask_dict:
            mask_dict[most_freq_mask] = {}
        
        # Add the gaze index and bbox to the dictionary under the most frequent mask
        mask_dict[most_freq_mask][gaze_indx] = {'bbox': bbox}
        
    
    if debug:
        print(mask_dict)
    
    return mask_dict

def generate_knowledge_graph( img_id, model = "gpt-4o", debug = True ):
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    results_df = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")
    
    bbox_dict = create_mask_dict( img_id, debug = debug )
        
    scene_description = results_df.loc[0, 'SCENE_DESCRIPTION']
    
    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system","content": [{"type": "text", "text": f"This is the knowledge your have about a scene: {scene_description}"}]},
        {"role": "user", "content": [{ "type": "text", "text": f"""Consider the following dictionary that contains a set of objects in the scene. 
                                    Analyze the spatial relationships between these objects based on their positions. 
                                    Analyze the properties of the objects (ex: shape, color, complexity, functionality, etc.
                                    Analyze the functionality of the objects.
                                    These are the bounding boxes {bbox_dict}"""}]}
    ],
    )
    
    
    answer = response.choices[0].message.content
    
    return answer

def generate_triples( img_id, model = "gpt-4o", debug = True ):
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    results_df = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")
        
    scene_description = results_df.loc[0, 'SCENE_DESCRIPTION']
    
    bbox_info = generate_knowledge_graph( img_id = img_id, debug = debug )   
    
    if debug:
        print(bbox_info)
    
    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system","content": [{"type": "text", "text": f"This is the knowledge your have about a scene: {scene_description}. This is what you know about the spatial relationships between objects in the scene: {bbox_info}"}]},
        {"role": "user", "content": [{ "type": "text", "text": f"""Please generate triples to build a scene graph for the following scene. 
                                    Use your visual understanding and prior knowledge to enrich the scene graph. 
                                    The triples should cover the following types of relationships:
                                    1. Spatial Relationships: Describing the position and arrangement of objects in the scene.
                                    2. Functional Relationships: Describing the purpose or use of the objects.
                                    3. Semantic Relationships: Describing meaningful connections between objects (e.g., objects that are commonly used together or associated with each other).
                                    4. Property Relationships: Describing the attributes of objects (e.g., shape, color, texture, material, etc.).
                                    5. Contextual Relationships: Describing the relationships between objects and the context of the scene.
                                    6. Distance Relationships: Describing the distance between objects.
                                    7. Size Relationships: Describing the size of objects.
                                    Please return only the triples in the format (subject, RELATION, object). Avoid including numbers in the triples.
                                    
                                    Example:
                                        (Slippers, are_on, Rug)
                                        (Table, is_under, Pendant light)
                                        (Sofa, is_near, Window)
                                        (Teapot, has_color, White)
                                        (Sofa, has_shape, Rectangular)
                                        (Rug, is_used_for, Adding warmth)
                                        (Cup, is_used_for, Drinking)
                                        (Cabinets, have_color, White)
                                        
                                    Please generate similar triples for the given scene. Return only the triples in the format (subject, RELATION, object)."""}]}
    ],
    )
    
    answer = response.choices[0].message.content
    answer = answer.replace("\n", "")
    
    for indx in results_df.index:
        results_df.loc[indx, 'BBOX_INFO'] = bbox_info
        results_df.loc[indx, 'SCENE_TRIPLES'] = answer
    
    results_df.to_csv(f"{HOME}/results/results_{img_id}_with_context.csv", index = False)
    
    
    return answer
    

def get_bbox_description( img_id ):
    
    file_path = f"{HOME}/results/results_{img_id}_with_context.csv"
    df = pd.read_csv(file_path)
    bbox_description = df.loc[0, 'BBOX_INFO']
    
    return bbox_description

def get_triples( img_id ):
    
    file_path = f"{HOME}/results/results_{img_id}_with_context.csv"
    df = pd.read_csv(file_path)
    triples = df.loc[0, 'SCENE_TRIPLES']
    triples_lst = triples.split(')  (')
    
    if len(triples_lst) == 1:
        triples_lst = triples_lst[0].split(')(')
    
    return triples_lst


def fix_triples( img_id ):
    
    file_path = f"{HOME}/results/results_{img_id}_with_context.csv"
    df = pd.read_csv(file_path)
    triples = df.loc[0, 'SCENE_TRIPLES']
    
    triples = triples.replace(")  ", ")")
    triples = triples.replace(")   (", ")  (")
    triples = triples.replace(")(", ")  (")
    
    # save the triples
    for indx in df.index:
        df.loc[indx, 'SCENE_TRIPLES'] = triples
    df.to_csv(file_path, index = False)
    
    return triples


def plot_knowledge_graph_from_triples( img_id ):


    file_path = f"{HOME}/results/results_{img_id}_with_context.csv"
    df = pd.read_csv(file_path)

    triples = df.loc[0, 'SCENE_TRIPLES']
    
    triples = triples.replace(")   (", ")  (")
    
    triples = triples.replace(")(", ")  (")
    
    # save the triples
    df.loc[0, 'SCENE_TRIPLES'] = triples
    df.to_csv(file_path, index = False)
    
    triples_lst = triples.split(')  (')
    
    
    if len(triples_lst) == 1:
        triples_lst = triples_lst[0].split(')(')

    G = nx.DiGraph()
    for triple in triples_lst:
        triple = triple.replace("(", "")
        triple = triple.replace(")", "")
        
        tokens = triple.split(', ')
        
        if len(tokens) == 3:
            subject, predicate, obj  = triple.split(', ')
            predicate = predicate.upper()
            
        if len(tokens) == 5:
            subject, predicate, obj, predicate2, obj2 = triple.split(', ')
            predicate = predicate.upper()
            predicate2 = predicate2.upper()
            G.add_edge(obj.strip(), obj2.strip(), label=predicate2.strip())
            edge_labels = nx.get_edge_attributes(G, 'label')
            
            
        G.add_edge(subject.strip(), obj.strip(), label=predicate.strip())
        edge_labels = nx.get_edge_attributes(G, 'label')

    pos = nx.spring_layout(G, k=1.2, iterations=100)

    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=3000, font_size=10, font_weight="bold", edge_color="gray")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red")

    #plt.title("Knowledge Graph Based on SCENE_TRIPLES")
    #plt.tight_layout()
    
    # save the graph
    plt.savefig(f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph.png", bbox_inches='tight')

    # save the graph in netwrokx to pkl
    with open(f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph.pkl", 'wb') as f:
        pickle.dump(G, f)
    
    plt.show()

def get_knowledge_graph( img_id ):
    
    with open(f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph.pkl", 'rb') as f:
        G = pickle.load(f)
        
    import networkx as nx
import pandas as pd
import pickle

def get_knowledge_graph_matrix(img_id):

    # Load the knowledge graph from a pickle file
    with open(f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph.pkl", 'rb') as f:
        G = pickle.load(f)
    
    # Extract the adjacency view from the graph
    adjacency_view = G.adj
    edges = []

    # Extract edges and weights (assuming there might be weights or you could use a count of 1 for unweighted graphs)
    for source, neighbors in adjacency_view.items():
        for target, info in neighbors.items():
            weight = info['label'] if 'label' in info else 1  # Use label or a dummy weight of 1
            edges.append({'Source': source, 'Target': target, 'Weight': weight})

    # Convert the list to a DataFrame
    edges_df = pd.DataFrame(edges)

    # Create a DataFrame for the adjacency matrix
    columns = list(set(edges_df['Source']).union(set(edges_df['Target'])))
    matrix_df = pd.DataFrame(0, columns=columns, index=columns)

    # Fill the matrix with weights
    for index, row in edges_df.iterrows():
        matrix_df.loc[row['Source'], row['Target']] = row['Weight']

    # Save the matrix to a CSV file
    matrix_df.to_csv(f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph_matrix.csv")

    return matrix_df

        
    return G

def convert_kg_to_neo4j( img_id ):
    
    file_path = f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph_matrix.csv"
    knowledge_graph_matrix = pd.read_csv(file_path)
    
    # Convert adjacency matrix to edge list format
    edge_list = []

    # Loop over rows and columns to extract edges with non-zero or non-empty relationships
    for source_node in knowledge_graph_matrix.index:
        for target_node in knowledge_graph_matrix.columns[1:]:  # Skip the first column header
            relationship = knowledge_graph_matrix.at[source_node, target_node]
            if relationship != 0 and relationship != '0':  # Check for actual relationships
                edge_list.append({
                    'Source': knowledge_graph_matrix.iloc[source_node, 0],  # First column is the row node label
                    'Target': target_node,
                    'Relationship': relationship
                })

    # Convert to DataFrame for ease of use
    edge_list_df = pd.DataFrame(edge_list)

    edge_list_file_path = f"{HOME}/results/Knowledge_Graph/{img_id}_knowledge_graph_matrix_neo4j.csv"
    edge_list_df.to_csv(edge_list_file_path, index=False)

    # Display the resulting edge list for inspection
    return edge_list_df



def convert_mc_to_neo4j(img_id):
    # Load the Markov Chain matrix
    file_path = f"{HOME}/results/Markov_Chains/{img_id}_markov_chain_matrix.csv"
    markov_chain_matrix = pd.read_csv(file_path)  # Load with first column as index
    markov_chain_matrix.index = markov_chain_matrix.columns
    
    # Convert adjacency matrix to edge list format
    edge_list = []

    # Loop over rows (source nodes) and columns (target nodes) to extract edges with non-zero relationships
    for source_node in markov_chain_matrix.index:
        for target_node in markov_chain_matrix.columns:
            
            # Ensure we get a single value from the matrix
            weight = markov_chain_matrix.loc[source_node, target_node]
            
            # Check if weight is a scalar and greater than 0
            if isinstance(weight, (int, float)):
                
                if weight > 0:
                    edge_list.append({
                            'From': source_node,
                        'To': target_node,
                        'Frequency': weight,
                })

    # Convert to DataFrame for easy handling and saving to CSV
    edge_list_df = pd.DataFrame(edge_list)


    # Save the edge list as a CSV file
    edge_list_file_path = f"{HOME}/results/Markov_Chains/{img_id}_markov_chain_matrix_neo4j.csv"
    edge_list_df.to_csv(edge_list_file_path, index=False)

    # Display the resulting edge list for inspection
    return edge_list_df



def get_markov_chain_pkl( img_id ):
    
    with open(f"{HOME}/results/Markov_Chains/{img_id}_markov_chain.pkl", 'rb') as f:
        G = pickle.load(f)
    
    adjacency_view = G.adj
    edges = []

    # Extract edges and weights
    for source, neighbors in adjacency_view.items():
        for target, info in neighbors.items():
            edges.append({'Source': source, 'Target': target, 'Weight': info['weight']})

    # Convert the list to a DataFrame
    edges_df = pd.DataFrame(edges)
    
    columns = list(set(set(edges_df['Source']).union(set(edges_df['Target']))))

    matrix_df = pd.DataFrame(0, columns = columns, index = columns)

    for indx, row in edges_df.iterrows():
        matrix_df.loc[row['Source'], row['Target']] = row['Weight']

    # convert all columns to float
    matrix_df = matrix_df.astype(float)

    # save the matrix
    matrix_df.to_csv(f"{HOME}/results/Markov_Chains/{img_id}_markov_chain_matrix.csv", index = False)
            
    return matrix_df

def get_markov_chain_matrix( img_id ):
    
    file_path = f"{HOME}/results/Markov_Chains/{img_id}_markov_chain_matrix.csv"
    df = pd.read_csv(file_path)
    
    return df

def plot_markov_chain_from_pkl(img_id):

    image_path = f"{HOME}/exp_images/{img_id}exp.jpg"
    pkl_path = f"{HOME}/results/Markov_Chains/{img_id}_markov_chain.pkl"

    # Load the background image
    img = plt.imread(image_path)

    # Load the graph
    with open(pkl_path, 'rb') as f:
        combined_graph = pickle.load(f)

    # Determine node positions based on node attribute (if stored) or manually set
    # If not stored, this will require modification to include node positions
    if 'pos' in nx.get_node_attributes(combined_graph, 'pos'):
        pos = nx.get_node_attributes(combined_graph, 'pos')
    else:
        # Assuming some default positions if not available - this might need to be adjusted
        pos = {node: (i * 0.1 % 1, i * 0.1 % 1) for i, node in enumerate(combined_graph.nodes())}

    # Set edge widths based on transition probabilities, adjusting scale factor for visibility
    edge_widths = [d['weight'] * 10 for _, _, d in combined_graph.edges(data=True)]  # Adjust the multiplier as needed

    # Plotting
    plt.figure(figsize=(12, 8))
    plt.imshow(img, extent=[0, 1, 0, 1], alpha=0.3)  # Set the image as a background
    nx.draw(combined_graph, pos, node_size=2000, node_color='lightblue', edge_color='gray', width=edge_widths,
            with_labels=True, font_weight='bold', arrowstyle='-|>', arrowsize=10)
    plt.title(f'Markov Chain Visualization for {img_id}')
    plt.axis('off')
    plt.show()


        
    return combined_graph


def plot_matrix_from_triples(img_id, cmap_color="viridis"):
    
    # Split triples into subject, relation, object
    triples = get_triples( img_id )
    
    data = [triple.replace('(', '').replace(')', '').split(', ') for triple in triples]

    # Create a dictionary to hold unique entities and relations
    unique_entities = defaultdict(int)
    relations = set()
    relation_counts = defaultdict(int)

    for subject, relation, obj in data:
        
        try:
            unique_entities[subject] += 1
            unique_entities[obj] += 1
            relations.add(relation)
            relation_counts[relation] += 1  # Increment relation count
        except:
            print(f"Error with triple: {triple}")
            break

    # Create a colormap for the heatmap and assign colors to relationships
    cmap = sns.color_palette(cmap_color, len(relations))
    
    entities = sorted(unique_entities.keys())
    relation_colors = {rel: cmap[i] for i, rel in enumerate(sorted(relations))}
    
    # Create a directional heatmap dataframe where only subject -> object triples are represented
    directional_heatmap_data = pd.DataFrame(0, index=entities, columns=entities)
    annotations = pd.DataFrame("", index=entities, columns=entities)

    # Fill the dataframe with relation codes (numeric) and track the annotations with relation names
    for subject, relation, obj in data:
        directional_heatmap_data.at[subject, obj] = list(relation_colors.keys()).index(relation) + 1
        annotations.at[subject, obj] = relation

    # Plot the heatmap for directional triples
    plt.figure(figsize=(12, 10))
    # Set mask for 0 values to be white
    mask = directional_heatmap_data == 0

    cmap_sns = sns.color_palette(cmap_color, len(relations))
    # Create the heatmap with white cells for zero values and horizontal grid lines
    ax = sns.heatmap(directional_heatmap_data, annot=True, cmap=cmap_sns, cbar=False, mask=mask, 
                    linecolor='white', linewidth=0.5)

    # Add horizontal grid lines
    ax.hlines(range(len(directional_heatmap_data) + 1), *ax.get_xlim(), color="#e5e7e9", linewidth=1)
    ax.vlines(range(len(directional_heatmap_data) + 1), *ax.get_ylim(), color="#e5e7e9", linewidth=1)

    # Add a title and rotate labels for readability
    #plt.title('Directional Heatmap of Object Relationships', fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)

    # Add labels for direction
    plt.xlabel("Object (Target)", fontsize=12)
    plt.ylabel("Object (Source)", fontsize=12)

    # Create a legend for the relationships
    # The legend corresponds to the color scheme for each relation
    for label, color in relation_colors.items():
        label_with_count = f"{label} ({relation_counts[label]})"
        plt.plot([], [], label=label_with_count, color=color, linewidth=15)

    # Display the legend
    plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1), title="Relationships", fontsize=10)

    # save the graph
    plt.savefig(f"{HOME}/results/Knowledge_Matrix/{img_id}_knowledge_matrix.png", bbox_inches='tight')

    # Show the heatmap
    plt.show()

# Function to convert a triple into a natural language sentence
def triple_to_sentence(triple):
    subject, predicate, obj = triple
    if predicate.startswith("is_"):
        # Handle "is_" predicates (e.g., is_on, is_under)
        predicate = predicate.replace("is_", "is ")
    elif predicate.startswith("are_"):
        # Handle "are_" predicates
        predicate = predicate.replace("are_", "are ")
    
    return f"The {subject.lower()} {predicate} the {obj.lower()}."


def ask_reasoning_question( triples, scene_description, query ):
    data = [triple.replace('(', '').replace(')', '').split(', ') for triple in triples]

    # Convert each triple to a natural language sentence
    natural_language_sentences = [triple_to_sentence(triple).replace("_", " ") for triple in data]
    sentence = " ".join(natural_language_sentences)

    client = OpenAI(api_key=OPENAI_API_KEY)

    # Use OpenAI GPT to answer the question
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system","content": [{"type": "text", "text": f"Consider the following knowledge base: {sentence}."}]},
            {"role": "user", "content": [{ "type": "text", "text": f"""Do not use colors, properties, or attributes in the locations. {query}. Do not use colors, properties, or attributes in the locations."""}]}
        ],
        temperature=0.0
        )
        
    answer = response.choices[0].message.content
    answer = answer.replace("\n", " ")
    answer = answer.replace("*", "")
    answer = answer.replace("*", " ")
        

    # Print the reasoning response from the LLM
    return answer



def is_plural(word1, word2):
    # Simple case where one word is just an "s" added
    if word1 + 's' == word2:
        return True
    # Words ending in "y", replace "y" with "ies"
    if word1.endswith('y') and word2 == word1[:-1] + 'ies':
        return True

    # Words ending in "ch", "sh", "x", "s", "z", add "es"
    if word1.endswith(('ch', 'sh', 'x', 's', 'z')) and word2 == word1 + 'es':
        return True
    
    return False


def compute_iou(bbox1, bbox2):
    x1, y1, x2, y2 = bbox1
    x1_, y1_, x2_, y2_ = bbox2
    
    # compute iou
    intersection = max(0, min(x2, x2_) - max(x1, x1_)) * max(0, min(y2, y2_) - max(y1, y1_))
    union = (x2 - x1) * (y2 - y1) + (x2_ - x1_) * (y2_ - y1_) - intersection
    iou = intersection / union
    return iou
    
    
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from collections import Counter

def is_plural(word1, word2):
    # Simple case where one word is just an "s" added
    if word1 + 's' == word2:
        return True
    # Words ending in "y", replace "y" with "ies"
    if word1.endswith('y') and word2 == word1[:-1] + 'ies':
        return True

    # Words ending in "ch", "sh", "x", "s", "z", add "es"
    if word1.endswith(('ch', 'sh', 'x', 's', 'z')) and word2 == word1 + 'es':
        return True
    
    return False



def compute_iou(bbox1, bbox2):
    x1, y1, x2, y2 = bbox1
    x1_, y1_, x2_, y2_ = bbox2
    
    # compute iou
    intersection = max(0, min(x2, x2_) - max(x1, x1_)) * max(0, min(y2, y2_) - max(y1, y1_))
    union = (x2 - x1) * (y2 - y1) + (x2_ - x1_) * (y2_ - y1_) - intersection
    iou = intersection / union
    return iou
    
    
import re
import collections

def exception_locations(location):
    
    if location == "Workspace desk":
        location = "Workspace Desk"
    
    location = location.replace("for Laundry Tasks", "Washing Machines")
    location = location.replace("For Laundry Tasks", "Washing Machines")
    location = location.replace("UNLIKELYST ", "")
    location = location.replace("LIKELYST ", "")
    location = location.replace("Near ", "")
    location = location.replace("Left ", "")
    location = location.replace("Right ", "")
    location = location.replace("Potted ", "")
    location = location.replace("Lights", "Light")
    location = location.replace("Pendant ", "")
    location = location.replace("String ", "")
    location = location.replace("Plants", "Plant")
    location = location.replace("Bedding", "Bed")
    
    location = location.replace("Wooden Chest ", "Chest")
    location = location.replace("Rugs", "Rug")
    location = location.replace("rug", "Rug")
    location = location.replace("Throw blanket", "Blanket")
    location = location.replace("Window Sill", "Window")
    location = location.replace("Windowsill", "Window")
    location = location.replace("Curtains", "Curtain")
    location = location.replace("Wall-mounted hook rack", "Hook rack")
    location = location.replace("Walls", "Wall")
    location = location.replace("Small box", "Box")
    
    location = location.replace("_", " ")
    location = location.replace(" Solution", "")
    location = location.replace("Storage unit", "Storage")
    #location = location.replace("Storage ", "")
    location = location.replace("Right ", "")
    location = location.replace("Loft ", "")
    location = location.replace("Office Chair", "Chair")
    location = location.replace("Blue ", "")
    location = location.replace("Green ", "")
    location = location.replace("White ", "")
    location = location.replace("Desk Chair", "Chair")
    location = location.replace("Fairy ", "")
    location = location.replace("Television", "TV")
    
    location = location.replace("bins", "Bins")
    location = location.replace("Sun-shaped ", "")
    location = location.replace("toys", "Toys")
    location = location.replace("toys", "Toys")
    location = location.replace("lamp", "Lamp")
    location = location.replace("pendant ", "")
    location = location.replace("One to three", "Floor")
    location = location.replace("artwork", "Artwork")
    location = location.replace("Child-created ", "")
    location = location.replace("Hanging fabric storage ", "")
    location = location.replace("bed", "Bed")
    location = location.replace("Twin ", "")
    location = location.replace("Plush ", "")
    location = location.replace("shelf", "Shelf")
    location = location.replace("Beds", "Bed")
    location = location.replace("pots", "Pots")
    location = location.replace("Copper ", "")
    location = location.replace(" perimeter", "")
    location = location.replace("Bedspread", "Bed")
    location = location.replace("pencils", "Pencils")
    location = location.replace("Colored ", "")
    location = location.replace("table", "Table")
    location = location.replace("Small ", "")
    location = location.replace("Toy tricycle", "Scooter")
    location = location.replace("toy", "Toy")
    location = location.replace("Colorful ", "")
    location = location.replace("Countertops", "Countertop")
    location = location.replace("Glass sliding door", "Door")
    location = location.replace("Refrigerator", "Fridge")
    location = location.replace("Knife block", "Knives")
    location = location.replace("Wooden Floor", "Floor")
    location = location.replace("Wooden ", "")
    location = location.replace("Wooden", "Floor")
    location = location.replace("flooring", "Floor")
    location = location.replace("Flooring", "Floor")
    location = location.replace("wall", "Wall")
    location = location.replace("crib", "Crib")
    location = location.replace("Mat", "Rug")
    location = location.replace("Guitars", "Guitar")
    location = location.replace("Telephone", "Smartphone")
    location = location.replace("Storage Basket", "Basket")
    location = location.replace("Bunk Bed", "Bed")
    
    location = location.replace("Upper Section", "Cabinets")
    location = location.replace("Tall ", "")
    location = location.replace("Close to ", "")
    location = location.replace("Open", "")
    location = location.replace("Wall Shelves", "Shelves")
    location = location.replace("room", "Room")
    location = location.replace("closet", "Closet")
    location = location.replace("Additional hanging space", "Hanging space")
    location = location.replace("End of Room", "Room")
    location = location.replace("BedRoom", "Bedroom")
    location = location.replace("Shelving", "Shelves")
    location = location.replace("Light brown Bedpread", "Bedpread")
    location = location.replace("Bedpread", "Bed")
    location = location.replace("Windows", "Window")
    location = location.replace("Paper Lantern", "Lantern")
    location = location.replace("Back ", "")
    location = location.replace("Accent ", "")
    location = location.replace("Armchair", "Chair")
    location = location.replace("Round ", "")
    
    return location
    

def process_location(location):

    location = exception_locations(location)
    
    if " and " in location:
        loc0 = location.split(" and ")[0]
        location = location.replace(location, loc0)
        
    location = re.sub(r"[^\w\s]", "", location).strip()
    
    return location

# Function to plot the data
def plot_rank(img_id, rank, data, target, title_label = "", show=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.bar(data.keys(), data.values(), color='skyblue', edgecolor='black')
    plt.xticks(rotation=90, fontsize=10)
    ax.set_xlabel('Locations', fontsize=12, weight='bold')
    ax.set_ylabel('Probability', fontsize=12, weight='bold')
    ax.set_title(f"[{rank} Ranking] Probability of {title_label} Locations for Target '{target}'", fontsize=14, weight='bold')
    plt.tight_layout()
    
    # save the plot
    plt.savefig(f"{HOME}/results/ranking_plots/IMG_{img_id}_rank_{rank}_{title_label}.png")
    
    if show:
        plt.show()
    plt.close(fig)

def generate_ranking_probabilities( img_id, list_type,  prob=True, plot=True ):

    # Load rankings and process locations
    most_likely, least_likely = load_rankings(img_id)
    target = get_target_object(IMG_ID)
    
    all_locations = get_locations(img_id)

    all_locations = [loc for loc in all_locations if "Left side" not in loc]
    all_locations = [loc for loc in all_locations if "Stainless steel" not in loc]
    all_locations = [loc for loc in all_locations if "Side" not in loc]

    all_locations = [process_location(loc) for loc in all_locations]
    all_locations = [loc for loc in all_locations if "Side" not in loc]
    
            
    if list_type == "most_likely":
        title_label = "Most Likely"
        trial_dict = most_likely
    
    if list_type == "least_likely":
        title_label = "Least Likely"
        trial_dict = least_likely

    # Standardize the locations in most_likely
    for trial in trial_dict.keys():
        trial_dict[trial] = [process_location(i) for i in trial_dict[trial]]

    # Extract locations based on their ranks
    location_ranks = {1: [], 2: [], 3: []}
    for trial in trial_dict.values():
        location_ranks[1].append(process_location(trial[0]))
        location_ranks[2].append(process_location(trial[1]))
        location_ranks[3].append(process_location(trial[2]))

    # Initialize location dictionaries
    location_counts = {rank: {loc: 0 for loc in all_locations} for rank in location_ranks}

    # Count occurrences of each location per rank
    for rank, loc_list in location_ranks.items():
        for loc in loc_list:
            if loc in location_counts[rank]:
                location_counts[rank][loc] += 1
            else:
                location_counts[rank][loc] = 1
                print(f"Added {loc} to locations_dic_{rank}")

    # Normalize the values for all ranks and sort
    for rank in location_counts:
        total_count = sum(location_counts[rank].values())
        if prob:
            location_counts[rank] = {k: v / total_count for k, v in location_counts[rank].items()}
        location_counts[rank] = dict(sorted(location_counts[rank].items(), key=lambda item: item[1], reverse=True))
        
        # remove "Left side"
        location_counts[rank] = {k: v for k, v in location_counts[rank].items() if ("side" not in k) or ("Side" not in k)}
        
    # Plot for each rank
    for rank in range(1, 4):
        
        rank_str = [ "1st" if rank == 1 else "2nd" if rank == 2 else "3rd" ][0]
        plot_rank(img_id, rank_str, location_counts[rank], target, title_label, plot)
        
    # save counts
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_{list_type}.pkl", "wb") as f:
        pickle.dump(location_counts, f)
        
    return location_counts

def get_target_object( img_id ):
    
    data = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")
    target = data[ data['IS_TARGET'] == 1]
    target_indx = target.index[0]
    target_label = target['MOST_FREQ_MASK'].values[0]
    
    return target_label

def get_locations( img_id ):
    
    target = get_target_object( img_id )
    
    triples = get_triples( img_id )
    
    locations = []
    for triple in triples:
        
        sub, rel, obj = triple.split(", ")
        
        locations.append( sub )
        
        if "_for" in rel:
            continue
        
        if "has_" in rel or "have" in rel:
            continue
        
        if "is_" in rel or "are_" in rel:
            locations.append( obj )
            
    locations = remove_duplicates( locations )
    
    locations = remove_target( locations, target )
    
    return locations

def remove_duplicates( locations ):
    
    locations = list(set(locations))
    
    return locations

def remove_target( locations, target ):
    
    locations = [loc for loc in locations if loc != target]
    
    return locations

# read excel the different sheets of file

def generate_qualitative_evaluation(  ):

    for img_id in range(1, 31):

        df = pd.DataFrame(columns=['Triple ID', 'Triple', 'Object Detection (1-6)',	'Relationship Accuracy (1-6)', 	'Attribute Accuracy (1-6)',	'Spatial and Positional Accuracy (1-6)',	'Functional Accuracy (1-6)',	'Overall Triple Plausibility (1-6)', 'Comments?'])
        
        triples = get_triples( img_id )
        
        for indx in range(0, len(triples)):
            
            df.loc[indx, 'Triple'] = triples[indx]
            df.loc[indx, 'Triple ID'] = indx
            df.loc[indx, 'Object Detection (1-6)'] = '1 = Strongly Disagree, 6 = Strongly Agree, N/A = Not Applicable'
            df.loc[indx, 'Relationship Accuracy (1-6)'] = '1 = Strongly Disagree, 6 = Strongly Agree, N/A = Not Applicable'
            df.loc[indx, 'Attribute Accuracy (1-6)'] = '1 = Strongly Disagree, 6 = Strongly Agree, N/A = Not Applicable'
            df.loc[indx, 'Spatial and Positional Accuracy (1-6)'] = '1 = Strongly Disagree, 6 = Strongly Agree, N/A = Not Applicable'
            df.loc[indx, 'Functional Accuracy (1-6)'] = '1 = Strongly Disagree, 6 = Strongly Agree, N/A = Not Applicable'
            df.loc[indx, 'Overall Triple Plausibility (1-6)'] = '1 = Strongly Disagree, 6 = Strongly Agree, N/A = Not Applicable'
            df.loc[indx, 'Comments?'] = ''
            
        df.to_excel(f"{HOME}/evaluation/triples/IMG_{img_id}.xlsx", index=False)



def process_ranking( img_id ):
    
    with open(f"{HOME}/results/ranking/IMG_{img_id}_most_likely.pkl", "rb") as f:
        most_likely = pickle.load(f)

    with open(f"{HOME}/results/ranking/IMG_{img_id}_least_likely.pkl", "rb") as f:
        least_likely = pickle.load(f)
        
    # Cleaned data
    processed_locations = {trial: [process_location(loc) for loc in loc_list] for trial, loc_list in most_likely.items()}

    # Count occurrences per rank (most likely: index 0, least likely: index 2)
    rankings = {0: [], 1: [], 2: []}
    for trial, loc_list in processed_locations.items():
        for idx, loc in enumerate(loc_list):
            rankings[idx].append(loc)

    # Count the occurrences for each rank
    rank_counts = {rank: collections.Counter(loc_list) for rank, loc_list in rankings.items()}

    # Plotting the data
    fig, axes = plt.subplots(3, 1, figsize=(10, 12))

    for i, (rank, counts) in enumerate(rank_counts.items()):
        axes[i].bar(counts.keys(), counts.values())
        axes[i].set_title(f'Locations for Rank {i+1}')
        axes[i].set_ylabel('Count')
        axes[i].set_xticklabels(counts.keys(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()
    
    # save the dictionaries into a pickle file
    with open(f"{HOME}/results/ranking_probabilities/IMG_{IMG_ID}_most_likely.pkl", "wb") as f:
        pickle.dump(likely_prob, f)

    with open(f"{HOME}/results/ranking_probabilities/IMG_{IMG_ID}_least_likely.pkl", "wb") as f:
        pickle.dump(unlikely_prob, f)
        
    return most_likely, least_likely


def load_rankings( img_id ):
    
    with open(f"{HOME}/results/ranking/IMG_{img_id}_most_likely.pkl", "rb") as f:
            most_likely = pickle.load(f)

    with open(f"{HOME}/results/ranking/IMG_{img_id}_least_likely.pkl", "rb") as f:
            least_likely = pickle.load(f)
            
    return most_likely, least_likely




def plot_bounding_boxes(image_filepath, csv_filepath, filter=None, debug=False):
    # Load the image
    image = cv2.imread(image_filepath)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB for plotting
    
    # Load the CSV file
    df = pd.read_csv(csv_filepath)
    
    overall_labels = {}
    temp = {}
    
    labels = []
    flag = False
    for indx in range(0, len(df)):
    
        # Calculate the most frequent label for each row across the GPT answer columns
        gpt_answer_columns = [df.loc[indx, f"BEST_MASK_GPTANSWER_{i}"] for i in range(10)]
        gpt_answer_columns = [col.replace('.', '') for col in gpt_answer_columns]
        
        labels_dict = dict(Counter(gpt_answer_columns))
        
        most_frequent_label = max(labels_dict, key=labels_dict.get)
        
        df.loc[indx, "MOST_FREQ_MASK"] = most_frequent_label
        
        # Get bounding box coordinates
        bbox = df.loc[indx, ['BEST_MASK_XMIN', 'BEST_MASK_XMAX', 'BEST_MASK_YMIN', 'BEST_MASK_YMAX']].values
        score = df.loc[indx, 'BEST_MASK_SCORE']
        area = df.loc[indx, 'BEST_MASK_AREA']
        temp = {'bbox': bbox, 'score': score, 'area': area}

        if area > 150000:
            continue

        for label in labels:
            if is_plural(label, most_frequent_label):
                most_frequent_label = label
                
            if is_plural(most_frequent_label, label):
                label = most_frequent_label
                overall_labels[label] = overall_labels.pop(most_frequent_label)
                flag = True
                
        if flag:
            labels = list(overall_labels.keys())
            flag = False
        
        # If the most frequent label is not in the overall_labels dictionary, add it
        if most_frequent_label not in overall_labels:
            overall_labels[most_frequent_label] = {}
            overall_labels[most_frequent_label][indx] = temp
        
        counter = 0
        # Compute IoU if the label already exists
        if most_frequent_label in overall_labels:
            
            for indx2 in overall_labels[most_frequent_label]:
                iou = compute_iou(bbox, overall_labels[most_frequent_label][indx2]['bbox'])
                
                # If the current box is contained in the overall box, skip
                if iou > 0.9:
                    continue
                else:
                    counter += 1
        
        if counter > 0:
            overall_labels[most_frequent_label][indx] = temp
        
    # Set up the plot
    plt.figure(figsize=(10, 10))
    plt.imshow(image_rgb)
    
    # Iterate through the filtered rows and plot bounding boxes
    for label, label_info in overall_labels.items():
        for indx, bbox_info in label_info.items():
            bbox = bbox_info['bbox']
            area = bbox_info['area']
            xmin, xmax, ymin, ymax = bbox
            if filter is None or label in filter:
                plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin], color='red', linewidth=2)
                plt.text((xmin+xmax)/2, ymin - 10, label, fontsize=14, color='red', backgroundcolor='white')
    
    # Plot gaze data
    gaze_data_X = df['GAZE_X'] 
    gaze_data_Y = df['GAZE_Y']
    
    plt.scatter(gaze_data_X, gaze_data_Y, c="blue", s=20)
    plt.grid(False)
    plt.axis('off')
    plt.show()
    
    return overall_labels



def plot_bounding_boxes_with_paletteb(image_filepath, csv_filepath, filter=None, debug=False, iou_threshold=0.9, fontsize=14, figsize=(10, 10)):
    # Load the image
    image = cv2.imread(image_filepath)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB for plotting
    
    # Load the CSV file
    df = pd.read_csv(csv_filepath)
    
    overall_labels = {}
    temp = {}
    
    labels = []
    flag = False
    color_index = 0
    label_colors = {}  # Dictionary to store color assigned to each unique label
    
    for indx in range(0, len(df)):
    
        # Calculate the most frequent label for each row across the GPT answer columns
        gpt_answer_columns = [df.loc[indx, f"BEST_MASK_GPTANSWER_{i}"] for i in range(10)]
        gpt_answer_columns = [col.replace('.', '') for col in gpt_answer_columns]
        
        labels_dict = dict(Counter(gpt_answer_columns))
        
        # normalise the label dict
        labels_dict = {k: v / sum(labels_dict.values()) for k, v in labels_dict.items()}
        
        most_frequent_label = max(labels_dict, key=labels_dict.get)
        
        most_frequent_label = df.loc[indx, "MOST_FREQ_MASK"] 
        
        try:
            prob = labels_dict[most_frequent_label]
        except:
            print(indx, most_frequent_label)
            break
        
        # Get bounding box coordinates
        bbox = df.loc[indx, ['BEST_MASK_XMIN', 'BEST_MASK_XMAX', 'BEST_MASK_YMIN', 'BEST_MASK_YMAX']].values
        score = df.loc[indx, 'BEST_MASK_SCORE']
        area = df.loc[indx, 'BEST_MASK_AREA']
        temp = {'bbox': bbox, 'score': score, 'area': area}

        #if area > 150000:
        #    continue

        for label in labels:
            if is_plural(label, most_frequent_label):
                most_frequent_label = label
                
            if is_plural(most_frequent_label, label):
                label = most_frequent_label
                overall_labels[label] = overall_labels.pop(most_frequent_label)
                flag = True
                
        if flag:
            labels = list(overall_labels.keys())
            flag = False
        
        # If the most frequent label is not in the overall_labels dictionary, add it
        if most_frequent_label not in overall_labels:
            overall_labels[most_frequent_label] = {}
            overall_labels[most_frequent_label][indx] = temp
        
        counter = 0
        # Compute IoU if the label already exists
        if most_frequent_label in overall_labels:
            
            for indx2 in overall_labels[most_frequent_label]:
                iou = compute_iou(bbox, overall_labels[most_frequent_label][indx2]['bbox'])
                
                # If the current box is contained in the overall box, skip
                if iou <= iou_threshold:
                    print(iou)
                    continue
                else:
                    print(counter)
                    counter += 1
        
        if counter > 0:
            overall_labels[most_frequent_label][indx] = temp

        # Assign a unique color from the palette to each label
        if most_frequent_label not in label_colors:
            label_colors[most_frequent_label] = color_palette[color_index % len(color_palette)]
            color_index += 1
    
    # Set up the plot
    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    
    # Iterate through the filtered rows and plot bounding boxes
    for label, label_info in overall_labels.items():
        color = label_colors[label]  # Get the assigned color for this label
        for indx, bbox_info in label_info.items():
            bbox = bbox_info['bbox']
            area = bbox_info['area']
            xmin, xmax, ymin, ymax = bbox
            if filter is None or label in filter:
                plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin], color=color, linewidth=2)
                plt.text((xmin+xmax)/2, ymin - 10, label + f" ({prob:.2f})", fontsize=fontsize, color='black', backgroundcolor=color, ha='center', alpha=0.9)

    # Plot gaze data
    gaze_data_X = df['GAZE_X'] 
    gaze_data_Y = df['GAZE_Y']
    
    plt.scatter(gaze_data_X, gaze_data_Y, c="red", s=50, alpha=0.7, marker='o', edgecolors='white', linewidths=0.5)
    plt.grid(False)
    plt.axis('off')
    
    # save the figure
    plt.savefig(f"{HOME}/results/bounding_boxes/IMG_{IMG_ID}_bbox.png", dpi=300)
    
    plt.show()
    
    return overall_labels


def plot_bounding_boxes_with_palette(image_filepath, csv_filepath, filter=None, debug=False, iou_threshold=0.9, fontsize=14, figsize=(10, 10)):
    # Load the image
    image = cv2.imread(image_filepath)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB for plotting
    
    # Load the CSV file
    df = pd.read_csv(csv_filepath)
    
    overall_labels = {}
    temp = {}
    
    labels = []
    color_index = 0
    label_colors = {}  # Dictionary to store color assigned to each unique label
    
    for indx in range(len(df)):
        # Get the most frequent label and bounding box information
        most_frequent_label = df.loc[indx, "MOST_FREQ_MASK"]
        bbox = df.loc[indx, ['BEST_MASK_XMIN', 'BEST_MASK_XMAX', 'BEST_MASK_YMIN', 'BEST_MASK_YMAX']].values
        score = df.loc[indx, 'BEST_MASK_SCORE']
        area = df.loc[indx, 'BEST_MASK_AREA']
        
        # Store bounding box details
        bbox_info = {'bbox': bbox, 'score': score, 'area': area}
        
        # Initialize label entry in overall_labels if not present
        if most_frequent_label not in overall_labels:
            overall_labels[most_frequent_label] = {}
        
        add_box = True  # Flag to check if the box should be added
        
        # Check IoU against existing boxes for this label
        for existing_indx, existing_info in overall_labels[most_frequent_label].items():
            iou = compute_iou(bbox, existing_info['bbox'])
            if iou >= iou_threshold:  # IoU is above threshold, do not add
                add_box = False
                break
        
        # If the box passed the IoU threshold check, add it
        if add_box:
            overall_labels[most_frequent_label][indx] = bbox_info

            # Assign a unique color if not already assigned
            if most_frequent_label not in label_colors:
                label_colors[most_frequent_label] = color_palette[color_index % len(color_palette)]
                color_index += 1
    
    # Set up the plot
    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    
    # Plot bounding boxes
    for label, label_info in overall_labels.items():
        color = label_colors[label]  # Get the assigned color for this label
        for indx, bbox_info in label_info.items():
            bbox = bbox_info['bbox']
            xmin, xmax, ymin, ymax = bbox
            if filter is None or label in filter:
                plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin], color=color, linewidth=2)
                plt.text((xmin + xmax) / 2, ymin - 10, label, fontsize=fontsize, color='black', backgroundcolor=color, ha='center', alpha=0.9)

    # Plot gaze data
    plt.scatter(df['GAZE_X'], df['GAZE_Y'], c="red", s=50, alpha=0.7, marker='o', edgecolors='white', linewidths=0.5)
    plt.grid(False)
    plt.axis('off')
    
    # Save the figure
    plt.savefig(f"{HOME}/results/bounding_boxes/IMG_{IMG_ID}_bbox.png", dpi=300)
    plt.show()
    
    return overall_labels



from PIL import Image, ImageDraw

def hex_to_rgba(hex_color, alpha=100):
    """
    Converts a hex color string to an RGBA tuple with specified transparency.
    """
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return (r, g, b, alpha)



def compute_iou(mask1, mask2):
    """
    Compute Intersection over Union (IoU) between two binary masks.
    """
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union != 0 else 0


### Load Model

## Gaze Driven Scene Segmentation - Full Dataset

### Prompting Segmentation with Gaze


**NOTE:** Process the image to produce an image embedding by calling `SAM2ImagePredictor.set_image`. `SAM2ImagePredictor` remembers this embedding and will use it for subsequent mask prediction. `SAM2ImagePredictor.predict` takes the following arguments:

- `point_coords` - `[np.ndarray or None]` - a `Nx2` array of point prompts to the model. Each point is in `(X,Y)` in pixels.
- `point_labels` - `[np.ndarray or None]` - a length `N` array of labels for the
point prompts. `1` indicates a foreground point and `0` indicates a
background point.
- `box` - `[np.ndarray or None]` - a length `4` array given a box prompt to the
model, in `[x_min, y_min, x_max, y_max]` format.
- `mask_input` - `[np.ndarray]` - a low resolution mask input to the model, typically coming from a previous prediction iteration. Has form `1xHxW`, where
for SAM, `H=W=256`.
- `multimask_output` - `[bool]` - if true, the model will return three masks.
For ambiguous input prompts (such as a single click), this will often
produce better masks than a single prediction. If only a single
mask is needed, the model's predicted quality score can be used
to select the best mask. For non-ambiguous prompts, such as multiple
input prompts, `multimask_output=False` can give better results.
- `return_logits` - `[bool]` - if true, returns un-thresholded masks logits
instead of a binary mask.
- `normalize_coords` - `[bool]` - if true, the point coordinates will be normalized to the range `[0,1]` and point_coords is expected to be wrt. image dimensions.

In [ ]:
predictor.set_image(image)

### Interactive Prompting with points

### Interactive Prompting with Boxes

## Generate Metadata

In [ ]:
from utils import *

## Label Masks

In [ ]:
def label_masks_with_gpt( img_id_min, img_id_max = None, mask_id_min = 0, mask_id_max = None ):
    
    img_id_max = len(metadata) if img_id_max is None else img_id_max
    for img_id in range(img_id_min, img_id_max):
    
        results_df_context = pd.read_csv(f"{HOME}/results/results_{img_id}_final.csv")
        results_df_no_context = pd.read_csv(f"{HOME}/results/results_{img_id}_final.csv")

        # Ensure that all BEST_MASK_GPTANSWER_* columns are cast to object dtype (string-compatible)
        answer_columns = [f"BEST_MASK_GPTANSWER_{ans_id}" for ans_id in range(10)]
        results_df_context[answer_columns] = results_df_context[answer_columns].astype(object)

        system_prompt = "You are an agent that describes images and classifies objects within that image.\nI will provide you an overall image of a scene. I will ask you to describe it with the highest possible detail. I will then show you segmented masks of this scene and I will ask you to classify them based on the context of the overall scene that I will initially present you with. Make sure that you remember the description of the overall scene for all the masks that I will ask you to classify. "
        scene_prompt = "Describe the image with the highest detail possible and remember your answer."
        mask_prompt = "The following image is a subset of the image that you just described. What's in this image? Answer one word only. "

        error = False

        mask_id_max = len(results_df_context) if mask_id_max is None else mask_id_max
        for indx in range(mask_id_min, mask_id_max):
            
            main_scene_link = results_df_context.loc[indx, 'IMG_LINK']
            mask_link = results_df_context.loc[indx, 'BEST_MASK_LINK']
            img_id = results_df_context.loc[indx, 'IMG_ID']
            
            answer_list = []
            print(f"IMG_ID {img_id} Indx {indx}" )
            for ans_id in range(0, 10):
                
                col_name = f"BEST_MASK_GPTANSWER_{ans_id}"
                try:
                    answer = classify_obj_with_context( main_scene_link, mask_link, system_prompt, scene_prompt, mask_prompt)
                    #print("\tAnswer", answer)
                    
                except:
                    print(f"\tError occurred for image {img_id} and index {indx} for answer {ans_id}")
                    print(f"\tMain Scene Link: {main_scene_link}")
                    print(f"\tMask Link: {mask_link}")
                    error = True
                    break
                results_df_context.loc[indx, col_name] = answer
            
            if error:
                break
            
        results_df_context.to_csv( f"{HOME}/results/results_{img_id}_with_context.csv", index = False)

In [ ]:

#targets = [ "Boots", "Shoes", "Shoe", "Sneakers", "Tennis"]
#targets = [ "Dollhouse", "Doll", "House", "Toyhouse", "Doll-house", "Doll house", "Toy House"]
#targets = ["Ball", "Football"]
targets = ["Basket"]

#targets = ["Basket", "Laundry"]

#targets = ["Stool" , "Chair"]

#targets = ["Heater"]

for img_id in [19]:
    
    img_no_target_path = f"{HOME}/exp_images/{img_id}back.jpg"
    img_no_target = load_image( img_no_target_path, show_image = False )
    link_no_mask = upload_img_to_server( f"{HOME}/exp_images/{img_id}back.jpg", CLIENT_ID)
    print(link_no_mask)

    results_df_context = pd.read_csv(f"{HOME}/results/results_{img_id}_with_context.csv")
    results_df_no_context = pd.read_csv(f"{HOME}/results/results_{img_id}_final.csv")

    # Ensure that all BEST_MASK_GPTANSWER_* columns are cast to object dtype (string-compatible)
    answer_columns = [f"BEST_MASK_GPTANSWER_{ans_id}" for ans_id in range(10)]
    results_df_context[answer_columns] = results_df_context[answer_columns].astype(object)

    system_prompt = "You are an agent that describes images and classifies objects within that image.\nI will provide you an overall image of a scene. I will ask you to describe it with the highest possible detail. I will then show you segmented masks of this scene and I will ask you to classify them based on the context of the overall scene that I will initially present you with. Make sure that you remember the description of the overall scene for all the masks that I will ask you to classify. "
    scene_prompt = "Describe the image with the highest detail possible and remember your answer."
    mask_prompt = "The following image is a subset of the image that you just described. What's in this image? Answer one word only. "

    error = False
    
    imn_answ = 0

    for indx in range(55, 60): #, len(results_df_context)):
        
        main_scene_link = results_df_context.loc[indx, 'IMG_LINK']
        mask_link = results_df_context.loc[indx, 'BEST_MASK_LINK']
        img_id = results_df_context.loc[indx, 'IMG_ID']
        
        results_df_context.loc[indx, 'IMG_LINK_NO_TARGET'] = link_no_mask
        results_df_context.loc[indx, 'IMG_PATH_NO_TARGET'] = img_no_target_path
        
        answer_list = []
        print(f"IMG_ID {img_id} Indx {indx}" )
        
        for ans_id in range(imn_answ, 10):
            
            col_name = f"BEST_MASK_GPTANSWER_{ans_id}"
            try:
                answer = classify_obj_with_context( main_scene_link, mask_link, system_prompt, scene_prompt, mask_prompt)
                
            except:
                print(answer)
                print(f"\tError occurred for image {img_id} and index {indx} for answer {ans_id}")
                print(f"\tMain Scene Link: {main_scene_link}")
                print(f"\tMask Link: {mask_link}")
                error = True
                break
            
            answer = answer.replace(".", "")
            results_df_context.loc[indx, col_name] = answer
            
            if answer in targets:
                results_df_context.loc[indx, 'IS_TARGET'] = 1
            else:
                results_df_context.loc[indx, 'IS_TARGET'] = 0
        
        if error:
            print(answer)
            break
        
        imn_answ = 0
    results_df_context.to_csv( f"{HOME}/results/results_{img_id}_with_context.csv", index = False)

In [ ]:
# correct participant ID

COND_ID = 1


for IMG_ID in range(5, 6):
    
    print(f"Processing image {IMG_ID}")

    # get partcipant ID
    participant_df = pd.read_csv(f"{HOME}/XSQ_Expt1_Data_2.csv")
    participant_df = participant_df[ (participant_df['ItemNum'] == IMG_ID) & (participant_df['Condition'] == COND_ID) ]
    participants = participant_df['ParticipantID'].tolist()

    participants =  pd.Series(participants)

    result = participants.groupby(participants).apply(lambda x: x.iloc[3:])
    output_list = result.tolist()

    results_df = pd.read_csv(f"{HOME}/results/results_{IMG_ID}_with_context.csv")

    results_df['PARTICIPANT_ID'] = output_list

    results_df.to_csv(f"{HOME}/results/results_{IMG_ID}_with_context.csv", index = False)


## Bounding Box Generation

In [ ]:
IMG_ID = 11

data_path = f"{HOME}/results/results_{IMG_ID}_with_context.csv"
experiment_df = pd.read_csv(data_path)

# rename the columns "GAZE_X" and "GAZE_Y" to "X" and "Y"
experiment_df = experiment_df.rename(columns={"GAZE_X": "X", "GAZE_Y": "Y"})

main_scene_path = f"{HOME}/exp_images/{IMG_ID}exp.jpg"

overall_labels = plot_bounding_boxes_with_palette(main_scene_path, data_path, iou_threshold = 0.7, figsize=(12, 12) )

## Markov Chain Generation

In [ ]:
IMG_ID = 26

data_path = f"{HOME}/results/results_{IMG_ID}_with_context.csv"
experiment_df = pd.read_csv(data_path)

# rename the columns "GAZE_X" and "GAZE_Y" to "X" and "Y"
experiment_df = experiment_df.rename(columns={"GAZE_X": "X", "GAZE_Y": "Y"})

main_scene_path = f"{HOME}/exp_images/{IMG_ID}exp.jpg"

generate_markov_chain( img_id = IMG_ID )

labels = plot_bounding_boxes_with_palette(main_scene_path, data_path, iou_threshold = 0.7, figsize=(12, 12) );

In [ ]:
# create an empty dataframe with columns "IMG_ID", "Semantic_Alignment", "MC_Edges", "Alignment_Percentage"
alignment_df = pd.DataFrame(columns = ["IMG_ID", "Semantic_Alignment", "MC_Edges", "KG_edges", "Alignment_Perct", "Alignment_Perct_Kg"])

indx = 0
for IMG_ID in range(1, 31):
    
    get_markov_chain_pkl(IMG_ID)
    
    get_knowledge_graph_matrix(IMG_ID)
    
    #get_knowledge_graph_pkl(IMG_ID)

    MC = convert_mc_to_neo4j( IMG_ID )
    KG = convert_kg_to_neo4j( IMG_ID )
    
    KG_aug = KG.copy()
    for i in range(0, len(KG)):
        source = KG.loc[i, 'Source']
        target = KG.loc[i, 'Target']
        relationship = KG.loc[i, 'Relationship']
        
        KG_aug.loc[len(KG) + i, 'Source'] = target
        KG_aug.loc[len(KG) + i, 'Target'] = source
        KG_aug.loc[len(KG) + i, 'Relationship'] = relationship

    # add a column named "Matched"
    KG_aug['Matched'] = False
    
    align = 0
    for i in range(0, len(KG_aug)):
        
        source_kg = KG_aug.loc[i, 'Source']
        target_kg = KG_aug.loc[i, 'Target']
        
        for j in range(0, len(MC)):
            source_mc = MC.loc[j, 'From']
            target_mc = MC.loc[j, 'To']
                    
            if source_mc in source_kg and target_mc in target_kg:
                KG_aug.loc[i, 'Matched'] = True
                align += 1
                

    alignment_df.loc[indx, "IMG_ID"] = IMG_ID
    alignment_df.loc[indx, "Semantic_Alignment"] = align
    alignment_df.loc[indx, "MC_Edges"] = len(MC)
    alignment_df.loc[indx, "Alignment_Perct"] = align / len(MC)
    alignment_df.loc[indx, "KG_edges"] = len(KG)/2
    alignment_df.loc[indx, "Alignment_Perct_Kg"] = align / (len(KG)/2)
    indx += 1
    
alignment_df.to_csv(f"{HOME}/results/alignment/semantic_alignment.csv", index = False)


## Knowledge Graph Generation

In [ ]:
IMG_ID = 1

for img_id in [IMG_ID]:
    
    triples = get_triples( img_id )
    
    plot_knowledge_graph_from_triples( img_id )
    
    plot_matrix_from_triples( img_id, cmap_color="Set2")

In [ ]:
FLAG = False

if FLAG:

    for img_id in range(30, 31):
        
        print(f"Image {img_id}")
        
        generate_image_description( img_id )

        print("\t", get_scene_description( 1 ))

        triples = generate_triples( img_id, debug=False )  
        
        triples = fix_triples( img_id )
        
        triples = get_triples( img_id )
        
        plot_knowledge_graph_from_triples( img_id )
        
        plot_matrix_from_triples( img_id, cmap_color="Set2")

## Semantic Reasoning


In [ ]:
ask_reasoning_question( triples, 
                    scene_description = description,
                    query = """Where in this image are you MOST likely to find the Slippers? 
                                Provide a ranked list of FIVE locations even if not explicitly stated in the knowledge base. 
                                Very briefly explain your reasoning.""" )


In [ ]:
import random

# run trials

trials = 50

num_loc_sample = 10

most_likely = {}
least_likely = {}
for IMG_ID in [15]:
    
    if IMG_ID == 22:
        continue
    
    print(f"Image {IMG_ID}")
    target = get_target_object( IMG_ID )
    all_locations = get_locations( IMG_ID )
    triples = get_triples( IMG_ID )
    
    print("\tTotal locations:", len(all_locations), "\tTarget:", target, "\n")
    
    most_likely[IMG_ID] = {}
    least_likely[IMG_ID] = {}
    for trial in range(0, trials):
        
        # randomly select five locations from all_locations
        locations = random.sample(all_locations, num_loc_sample)
        ranked_locations = ask_reasoning_question( triples, scene_description = description, 
                                                query =   f"""Given these locations {locations} which are the MOST and LEAST likely places to find {target}? 
                                        1. Provide a ranked list of THREE locations for each. 
                                        2. Ensure you ALWAYS provide a ranked list of THREE locations for MOST LIKELY.
                                        3. Ensure you ALWAYS provide a ranked list of THREE locations for LEAST LIKELY.
                                        4. Do not use conjunction e.g. "AND", "OR", "NOT", "WITH", etc. Example of NOT allowed, "Sofa AND Coffee Table", "Books AND Magazines", "Dresser WITH mirror".
                                        5. Do not use quotation marks or string quotes.
                                        6. Do not use attributes with the locations. Example of NOT allowed: "Green dresser", "Dresser on wheels", "Flower Rug" etc.
                                        7. Do not use "Solution" or "Storage" in the locations. Example of NOT allowed: "Solution", "Storage"
                                        8. Do not use Properties in the locations. Example of NOT allowed: "Blue Chair", "White Desk", "Green Dresser", "Flower Rug"
                                        9. Do not use COLORS in the locations. Example of NOT allowed: "Red Chair", "Blue Dresser", "Green Rug"
                                        10. The same location cannot be in both lists.
                                        11. The output should have the format LIKEST: [location1, location2, location3]; UNLIKELIEST: [location1, location2, location3]""" )
        
        print(f"\tTrial {trial+1}: {ranked_locations}")
        
        most_likely[IMG_ID][trial+1] = ranked_locations.replace("_", " ")
        most_likely[IMG_ID][trial+1] = ranked_locations.replace("Blue Chair", "Chair")
        most_likely[IMG_ID][trial+1] = ranked_locations.split(";")[0]
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("LIKELIEST: ", "")
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("LIKEST: ", "")
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("LIKELY: ", "")
        
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("[", "").replace("]", "")
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace(".", "")
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("_", " ")
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].strip()
        
        if " and " in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].split(" and ")[1]
            
        if " on " in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].split(" on ")[0]
            
        if " with " in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].split(" with ")[0]
            
        if " of " in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].split(" of ")[1]
            
        if "Green dresser" in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("Green dresser", "Dresser")
            
        if "Flower rug" in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("Flower rug", "Rug")
            
        if "White crib" in most_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {most_likely[IMG_ID][trial+1]}")
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].replace("White crib", "Crib")
            
        most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial+1].split(", ")
        
        
        least_likely[IMG_ID][trial+1] = ranked_locations.replace("_", " ")
        least_likely[IMG_ID][trial+1] = ranked_locations.replace("Blue Chair", "Chair")
        least_likely[IMG_ID][trial+1] = ranked_locations.split(";")[1]
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace(" UNLIKELIEST: ", "")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace(" UNLIKELYST: ", "")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("UNLIKELIEST: ", "")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("UNLIKELY: ", "")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("[", "").replace("]", "")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace(".", "")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("_", " ")
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].strip()
        
        if " and " in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].split(" and ")[1]
        
        if " on " in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].split(" on ")[0]
            
        if " with " in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].split(" with ")[0]
            
        if "Green dresser" in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("Green dresser", "Dresser")
            
        if "Flower rug" in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("Flower rug", "Rug")
            
        if "White crib" in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].replace("White crib", "Crib")
        
        if " of " in least_likely[IMG_ID][trial+1]:
            print(f"\tTrial {trial+1}: {least_likely[IMG_ID][trial+1]}")
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].split(" of ")[1]
        
        least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial+1].split(", ")
        
        
        if len(most_likely[IMG_ID][trial+1]) != 3:
            # copy the previous trial
            most_likely[IMG_ID][trial+1] = most_likely[IMG_ID][trial]
            
        if len(least_likely[IMG_ID][trial+1]) != 3:
            least_likely[IMG_ID][trial+1] = least_likely[IMG_ID][trial]
            
        
        print( f"\tMost likely:\t{most_likely[IMG_ID][trial+1]}")
        print(f"\tLeast likely:\t{least_likely[IMG_ID][trial+1]}")
        
        print("\n")
        
    print("-----------------------------------\n")
    
    # save the dictionaries into a pickle file
    with open(f"{HOME}/results/ranking/IMG_{IMG_ID}_most_likely.pkl", "wb") as f:
        pickle.dump(most_likely, f)
    
    with open(f"{HOME}/results/ranking/IMG_{IMG_ID}_least_likely.pkl", "wb") as f:
        pickle.dump(least_likely, f)
        
        
# fixing dictionaries
for IMG_ID in [15]:
    
    if IMG_ID == 22:
        continue

    # open the pickle files
    with open(f"{HOME}/results/ranking/IMG_{IMG_ID}_most_likely.pkl", "rb") as f:
        most_likely = pickle.load(f)

    with open(f"{HOME}/results/ranking/IMG_{IMG_ID}_least_likely.pkl", "rb") as f:
        least_likely = pickle.load(f)
        
    most_likely = most_likely[IMG_ID]
    least_likely = least_likely[IMG_ID]
    
    # write
    with open(f"{HOME}/results/ranking/IMG_{IMG_ID}_most_likely.pkl", "wb") as f:
        pickle.dump(most_likely, f)
    
    with open(f"{HOME}/results/ranking/IMG_{IMG_ID}_least_likely.pkl", "wb") as f:
        pickle.dump(least_likely, f)
        


In [ ]:
IMG_ID = 1

likely_prob = generate_ranking_probabilities( IMG_ID, "most_likely" )
unlikely_prob = generate_ranking_probabilities( IMG_ID, "least_likely" )


## Statistical Analysis & Significance Testing

In [ ]:

import numpy as np
from scipy.stats import entropy
from scipy.stats import wasserstein_distance
from scipy.stats import mannwhitneyu
import pandas as pd

from scipy.special import rel_entr

# ignore Runtime Warning
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


def load_ranking_counts(img_id, type):
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_{type}.pkl", "rb") as f:
        counts = pickle.load(f)
    return counts

def update_label(img_id, counts, old_label, new_label, type):
    
    for ranking in [1,2,3]:
        # update the key of old_label to new_label
        counts[ranking][new_label] = counts[ranking].pop(old_label)
    
    # sort the dictionary descending
    for ranking in [1,2,3]:
        counts[ranking] = {k: v for k, v in sorted(counts[ranking].items(), key=lambda item: item[1], reverse=True)}
        
    # save the counts
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_{type}.pkl", "wb") as f:
        pickle.dump(counts, f)
    
    return counts

def update_ranking_counts(img_id, counts, label_to_delete, label_to_update, type):

    for ranking in [1,2,3]:
        value = counts[ranking][label_to_delete]
        counts[ranking].pop(label_to_delete)
        counts[ranking][label_to_update] += value
        
        # sort the dictionary descending
        counts[ranking] = {k: v for k, v in sorted(counts[ranking].items(), key=lambda item: item[1], reverse=True)}
    
    # save the counts
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_{type}.pkl", "wb") as f:
        pickle.dump(counts, f)
        
    return counts

def delete_label(img_id, counts, label, type):
    
    for ranking in [1,2,3]:
        counts[ranking].pop(label)
    
    # save the counts
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_{type}.pkl", "wb") as f:
        pickle.dump(counts, f)
        
    return counts

def key_normalisation(img_id, counts_likely, counts_unlikely):
    
    all_keys = []
    for rank in [1,2,3]:
        all_keys.extend( list(counts_likely[rank].keys()) )
        all_keys.extend( list(counts_unlikely[rank].keys()) )
        
    all_keys = list(set(all_keys))
    
    for key in all_keys:
        
        for ranking in [1,2,3]:
            
            # if key is not in counts_likely[ranking], add it with value 0
            if not key in counts_likely[ranking].keys():
                counts_likely[ranking][key] = 0
                
            if not key in counts_unlikely[ranking].keys():
                counts_unlikely[ranking][key] = 0
    
    # save the counts
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_most_likely.pkl", "wb") as f:
        pickle.dump(counts_likely, f)
    
    with open(f"{HOME}/results/ranking_probabilities/IMG_{img_id}_Counts_least_likely.pkl", "wb") as f:
        pickle.dump(counts_unlikely, f)
    
    return counts_likely, counts_unlikely
    

def compute_jensen_shannon_dict(dist1, dist2):
    """
    Compute Jensen-Shannon divergence between two probability distributions.
    
    Args:
        dist1 (dict): First probability distribution as a dictionary {label: probability}
        dist2 (dict): Second probability distribution as a dictionary {label: probability}
    
    Returns:
        float: Jensen-Shannon divergence value
    """
    # Get all unique keys
    all_keys = set(dist1.keys()).union(set(dist2.keys()))
    
    # Convert distributions to numpy arrays with 0s for missing keys
    p = np.array([dist1.get(key, 0) for key in all_keys])
    q = np.array([dist2.get(key, 0) for key in all_keys])
    
    # Normalize if not already normalized
    p = p / np.sum(p)
    q = q / np.sum(q) 
    
    # Calculate the average distribution
    m = 0.5 * (p + q)
    
    # Calculate JSD using scipy's implementation
    # JSD = 0.5 * (KL(P||M) + KL(Q||M))
    jsd = 0.5 * (entropy(p, m) + entropy(q, m))
    
    return jsd



def compute_jensen_shannon(p, q):
    """ Compute Jensen-Shannon divergence between two probability distributions. """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    
    return np.sqrt(0.5 * (np.sum(rel_entr(p, m)) + np.sum(rel_entr(q, m))))


def calculate_metrics(dist1, dist2):
    """ Compute JSD and Wasserstein distance between two distributions. """
    keys = set(dist1).union(set(dist2))
    p = np.array([dist1.get(key, 0) for key in keys])
    q = np.array([dist2.get(key, 0) for key in keys])
    
    jsd = compute_jensen_shannon(p, q, )
    
    return jsd


def permutation_test_jsd(dist1, dist2, num_permutations=5000):
    keys = list(set(dist1.keys()).union(set(dist2.keys())))
    original_jsd = compute_jensen_shannon([dist1.get(key, 0) for key in keys], [dist2.get(key, 0) for key in keys])
    
    combined_values = np.array([dist1.get(key, 0) for key in keys] + [dist2.get(key, 0) for key in keys])
    greater_count = 0
    
    for _ in range(num_permutations):
        np.random.shuffle(combined_values)
        new_dist1_values = combined_values[:len(keys)]
        new_dist2_values = combined_values[len(keys):]
        
        new_jsd = compute_jensen_shannon(new_dist1_values, new_dist2_values)
        if new_jsd >= original_jsd:
            greater_count += 1
    
    p_value = greater_count / num_permutations
    return p_value


# Assuming experiment_results_jsd is your DataFrame containing p-values
def highlight_significant_pvals(val):
    """
    Highlights the cells in the DataFrame with a green background if p-value < 0.05.
    """
    color = 'green' if val < 0.05 else ''
    return f'background-color: {color}'
    


def apply_styles_based_on_pvalues(data_df, pvalues_df):
    """
    Apply styles to data_df based on the significance values in pvalues_df.
    """
    def style_func(val, index, col):
        if pvalues_df.at[index, col] < 0.05:
            return 'background-color: green'
        return ''

    # Apply the style function to each cell in the DataFrame
    styled_df = data_df.style.apply(lambda x: [style_func(val, x.name, x.index[i]) for i, val in enumerate(x)], axis=1)
    return styled_df


